> **Research record.** Cleaned copy of `800-research-users-locations/3302-third-places-macro-micro-ddd.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Neighbourhood-level and visit-level triple-difference estimates of third-place effects.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

# 3302 — Third-place activation: macro-to-micro continuous DDD

**Question.** After the onset of remote work, did Gush Dan home statistical areas
(2022) with a higher *continuous* intensity of detected third places experience
a stronger shift from out-of-neighbourhood trips toward local POI time — and does
distance to the CBD, and occupational work-from-home capacity, moderate that dose?

This notebook is a **macro-to-micro** companion to
[`05-neighborhood-retention.ipynb`](05-neighborhood-retention.ipynb).
Step 4 estimates five nested WLS models: a baseline donut (Model 1), occupational
WFH capacity (Model 2), amenity pull (Model 3), an additive horse race of WFH vs
amenities (Model 4), and a WFH × amenity synergy test without Dist (Model 5).

| Decision | Choice |
|---|---|
| Unit | `yishuv_stat_2022` (home SA of workers with a known work location) |
| Clock | weekday (Sun–Thu), **daytime** 06:00–22:00; duration by overlap |
| Local radius | `dist_to_home < OUTBOUND_M` (1 km) |
| Pre | January 2020 |
| Post | 2023Q1 only (Jan–Mar); `Post = 1` iff `year_quarter == 2023Q1` |
| Dist | `dist_to_cbd_km`; WLS uses $\ln(1+\mathrm{Dist}_i)$ (`np.log1p`) |
| WFH | `WFH_pot` = `it_pct` (CBS 2022; missing → 0), entered **continuously** in Models 2–5 |
| Doses | **Main WLS:** `IHS_Cand`, `IHS_TP`, `IHS_ATM` (`log(1+` home-geohash 1 km mean`)`) and `TP_Ratio` = `TP_n / (Cand_n + 1)`. Raw means stay on `df_sa`. **Appendix:** polygon-halo log1p (`IHS_*_sa`). Do not horse-race home-disc vs polygon in one equation. |
| Estimator | `smf.wls`; weights = `n_workers_pre`; cluster at SA; **SA FE** (`C(yishuv_stat_2022)`; subsumes locality) |
| Time FE | `C(year_quarter)` (2020Q1 vs 2023Q1; collinear with `Post`) |

**Outcomes (each fit separately).** Eight overlapping duration shares (they do
**not** partition the day) plus three SA means of user-level mobility:

1. `share_local_all_h` — local hours / all hours (spatial retention)
2. `share_home_nonwork_h` — home non-work hours / non-work hours (`flag_home AND NOT flag_work`) / `NOT flag_work`
3. `share_local_nothome_nonwork_h` — local not-home hours / not-home & not-work hours
4. `share_poi_all_h` — named-POI hours / all hours
5. `share_local_poi_h` — local POI hours / all POI hours
6. `share_poi_of_local_h` — local POI hours / all local hours
7. `share_local_cand_h` — local candidate hours / all candidate hours
8. `share_tp_poi_h` — detected-TP hours / all POI hours
9. `rg_mean` — SA mean radius of gyration, all daytime dwells (m)
10. `rg_poi_mean` — SA mean radius of gyration, named POIs only (m)
11. `entropy_local_mean` — SA mean local Shannon entropy (1 km, named POIs)

Requires Part 1 of 3300 (`COV_TABLE`). Set `REBUILD_* = False` after the first
successful BigQuery write.


In [ ]:
import os
import sys
import warnings
from pathlib import Path

_proj_dir = os.path.join(sys.prefix, "share", "proj")
os.environ.setdefault("PROJ_DATA", _proj_dir)
os.environ.setdefault("PROJ_LIB", _proj_dir)
try:
    from pyproj import datadir as _pyproj_datadir
    _pyproj_datadir.set_data_dir(_proj_dir)
except Exception:
    pass

project_root = Path("../../..")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
_nb_dir = Path.cwd()
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

from geofunctions import utils

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid", context="paper")
rng = np.random.default_rng(20260819)

In [ ]:
# ---------------------------------------------------------------------------
# Config (copied from 3300; 3302-specific tables and flags at the bottom)
# ---------------------------------------------------------------------------
CENSUS_TABLE = "phd-habidatum.phd_census.all_sa_main_features"
CENSUS_2022_TABLE = "phd-habidatum.phd_census.census_2022"
CENSUS_OCC_COLS = [
    "it_pct",
    "finance_pct",
    "academic_occup_pct",
    "academic_degree_pct",
]
PU_TABLE = "phd-habidatum.phd_models_results.google_3rd_places_detected_pu_20260601"
DWELLS_TABLE = "phd-habidatum.phd_dwells.dwells_pois_home_work_v2"
PROFILES_TABLE = "phd-habidatum.phd_datasets.user_profiles_monthly"
WORK_TABLE = "phd-habidatum.phd_users_locations.work_locations"
GH_TABLE = "phd-habidatum.phd_adm.geohashes_with_sa_gushdan"  # geohash, yishuv_stat_2022, geometry
COV_TABLE = "phd-habidatum.phd_datasets.sa_retention_covariates_3300"
TEL_AVIV_SEMEI = 5000  # CBS locality code; builds flag_tel_aviv for Step 3b Dist vs TA overlap

DETECT_BUFFER_M = 500  # SA-polygon halo (appendix); not the home-disc radius
OUTBOUND_M = 1000  # local accessibility: dist_to_home and home-geohash walkable stock
WORK_NEAR_M = 250
MIN_WORKERS = 10
MIN_WORKER_DAYS = 40
MIN_WINDOW_COVERAGE = 0.10
MIN_POI_DWELLS_PER_MONTH = 5
ENTROPY_RADIUS_M = 1000  # local spatial entropy neighbourhood

PRE_MONTH = "2020-01-01"
POST_MONTHS = ("2023-01-01", "2023-02-01", "2023-03-01")
DID_MONTHS = (PRE_MONTH, *POST_MONTHS)

# Time-of-day windows (local Israel clock). 3302 uses daytime only.
WINDOWS = {
    "all24": (0, 24),
    "daytime": (6, 22),
    "morning": (6, 9),
    "midday": (9, 16),
    "break": (11, 15),
    "evening": (16, 22),
}
PRIMARY_WINDOW = "daytime"
DAYTIME = WINDOWS["daytime"]

CIVIC_CAND_TYPES = ("museum", "library", "community_center", "cultural_center")

# First-of-month dates, H2 2019 (Jul) through Q3 2023 (Sep).
QUARTER_MONTHS = tuple(
    pd.Timestamp(y, m, 1).strftime("%Y-%m-%d")
    for y in range(2019, 2024)
    for m in range(1, 13)
    if pd.Timestamp(2019, 7, 1) <= pd.Timestamp(y, m, 1) <= pd.Timestamp(2023, 9, 1)
)

OUTCOME_TABLE_3302 = "phd-habidatum.phd_datasets.sa_retention_outcomes_q_3302"
MOBILITY_TABLE_3302 = "phd-habidatum.phd_datasets.sa_retention_user_mobility_3302"
MOBILITY_PARQUET = _nb_dir / "cache" / "3302_user_mobility.parquet"

# Set False after the first successful BQ / parquet write.
REBUILD_QUARTERLY = True
REBUILD_MOBILITY = True

SHARE_OUTCOMES = [
    "share_local_all_h",
    "share_home_nonwork_h",
    "share_local_nothome_nonwork_h",
    "share_poi_all_h",
    "share_local_poi_h",
    "share_poi_of_local_h",
    "share_local_cand_h",
    "share_tp_poi_h",
]
SHARE_LABELS = {
    "share_local_all_h": "Local hours / all hours",
    "share_home_nonwork_h": "Home non-work hours / non-work hours",
    "share_local_nothome_nonwork_h": "Local not-home hours / not-home & not-work hours",
    "share_poi_all_h": "POI hours / all hours",
    "share_local_poi_h": "Local POI hours / all POI hours",
    "share_poi_of_local_h": "Local POI hours / all local hours",
    "share_local_cand_h": "Local candidate hours / all candidate hours",
    "share_tp_poi_h": "Detected-TP hours / all POI hours",
}
MOBILITY_OUTCOMES = ["rg_mean", "rg_poi_mean", "entropy_local_mean"]
GYRATION_OUTCOMES = ["rg_mean", "rg_poi_mean"]
MOBILITY_LABELS = {
    "rg_mean": "Radius of gyration, all dwells (m)",
    "rg_poi_mean": "Radius of gyration, named POIs (m)",
    "entropy_local_mean": "Local spatial entropy (nats)",
}
ALL_OUTCOMES = SHARE_OUTCOMES + MOBILITY_OUTCOMES
OUTCOME_LABELS = {**SHARE_LABELS, **MOBILITY_LABELS}
DOSE_COLS = ("IHS_Cand", "IHS_TP", "IHS_ATM", "TP_Ratio")  # WLS amenity doses
DOSE_COLS_N = ("Cand_n", "TP_n", "ATM_n")  # raw means on df_sa only; not in regressions
DOSE_COLS_SA = ("IHS_Cand_sa", "IHS_TP_sa", "IHS_ATM_sa")  # log1p of SA-polygon halo

## Helpers

Main WLS doses are `log(1 +` SA-mean venue count within 1 km of January 2020
home geohashes`)`: `IHS_Cand`, `IHS_TP`, `IHS_ATM`. Raw means are `Cand_n`,
`TP_n`, `ATM_n`. Polygon-halo log1p versions (`IHS_*_sa`) are appendix
robustness. No density and no population scaling. CI helper is the normal
interval on a worker-weighted SA mean (duration shares are continuous, not
binomial counts).


In [ ]:
def attach_intensities(sa):
    """Home-disc walkable stock (main) and SA-polygon halo (appendix).

    SQL → Python:
      n_candidate / n_detected / n_atm  →  Cand_n / TP_n / ATM_n
      IHS_* = log1p of those means (WLS doses).
    """
    sa = sa.copy()
    sa["Cand_n"] = pd.to_numeric(sa["n_candidate"], errors="coerce").fillna(0.0)
    sa["TP_n"] = pd.to_numeric(sa["n_detected"], errors="coerce").fillna(0.0)
    sa["ATM_n"] = pd.to_numeric(sa["n_atm"], errors="coerce").fillna(0.0)
    sa["IHS_Cand"] = np.log1p(sa["Cand_n"].astype(float))
    sa["IHS_TP"] = np.log1p(sa["TP_n"].astype(float))
    sa["IHS_ATM"] = np.log1p(sa["ATM_n"].astype(float))
    sa["TP_Ratio"] = sa["TP_n"].astype(float) / (sa["Cand_n"].astype(float) + 1.0)
    sa["Cand_n_med"] = pd.to_numeric(sa["n_candidate_med"], errors="coerce")
    sa["TP_n_med"] = pd.to_numeric(sa["n_detected_med"], errors="coerce")
    sa["ATM_n_med"] = pd.to_numeric(sa["n_atm_med"], errors="coerce")
    sa["IHS_Cand_med"] = np.log1p(sa["Cand_n_med"].astype(float))
    sa["IHS_TP_med"] = np.log1p(sa["TP_n_med"].astype(float))
    sa["IHS_ATM_med"] = np.log1p(sa["ATM_n_med"].astype(float))
    sa["Cand_n_sa"] = pd.to_numeric(sa["n_candidates_sa"], errors="coerce").fillna(0.0)
    sa["TP_n_sa"] = pd.to_numeric(sa["n_detected_sa"], errors="coerce").fillna(0.0)
    sa["ATM_n_sa"] = pd.to_numeric(sa["n_atm_sa"], errors="coerce").fillna(0.0)
    sa["IHS_Cand_sa"] = np.log1p(sa["Cand_n_sa"].astype(float))
    sa["IHS_TP_sa"] = np.log1p(sa["TP_n_sa"].astype(float))
    sa["IHS_ATM_sa"] = np.log1p(sa["ATM_n_sa"].astype(float))
    sa["n_homes_dose"] = pd.to_numeric(sa.get("n_homes_dose"), errors="coerce").fillna(0)
    sa["Dist"] = pd.to_numeric(sa["dist_to_cbd_km"], errors="coerce")
    return sa


def attach_local_volume(df):
    """Absolute local daytime hours and per-worker / log1p transforms."""
    d = df.copy()
    if "dur_local_all_h" not in d.columns:
        d["dur_local_all_h"] = (
            pd.to_numeric(d["share_local_all_h"], errors="coerce")
            * pd.to_numeric(d["dur_total"], errors="coerce")
        )
    else:
        d["dur_local_all_h"] = pd.to_numeric(d["dur_local_all_h"], errors="coerce")
    n = pd.to_numeric(d["n_workers"], errors="coerce").replace(0, np.nan)
    d["local_h_per_worker"] = d["dur_local_all_h"] / n
    d["log_local_h_per_worker"] = np.log1p(d["local_h_per_worker"])
    return d


def type_in_sql(col, types):
    types_sql = ", ".join(f"'{t}'" for t in types)
    return (
        "REGEXP_REPLACE(LOWER(REPLACE(REPLACE(COALESCE("
        f"{col}, ''), '-', '_'), ' ', '_')), r'[^a-z0-9_]', '') IN ({types_sql})"
    )


def weighted_mean_ci(df, group_cols, value_col, weight_col="n_workers"):
    """Worker-weighted mean with approximate 95% normal CI (effective n)."""
    rows = []
    for keys, g in df.dropna(subset=[value_col, weight_col]).groupby(group_cols, observed=True):
        if not isinstance(keys, tuple):
            keys = (keys,)
        w = g[weight_col].astype(float).to_numpy()
        x = g[value_col].astype(float).to_numpy()
        wsum = float(w.sum())
        if wsum <= 0:
            continue
        mu = float(np.average(x, weights=w))
        var = float(np.average((x - mu) ** 2, weights=w))
        n_eff = (wsum ** 2) / float(np.sum(w ** 2))
        se = np.sqrt(var / max(n_eff, 1.0))
        row = dict(zip(group_cols, keys))
        row.update(
            mean=mu, n=len(g), n_eff=n_eff,
            ci_low=mu - 1.96 * se, ci_high=mu + 1.96 * se,
        )
        rows.append(row)
    return pd.DataFrame(rows)


def to_plot_period(ts):
    """Quarter label; January 2020 is its own tick (not pooled with Feb–Mar 2020)."""
    ts = pd.Timestamp(ts)
    if ts.year == 2020 and ts.month == 1:
        return "Jan 2020"
    if ts.year == 2020 and ts.month in (2, 3):
        return "Feb–Mar 2020"
    return str(ts.to_period("Q"))


def period_sort_key(label):
    """Sort: 2019Q3, 2019Q4, Jan 2020, Feb–Mar 2020, 2020Q2, …"""
    if label == "Jan 2020":
        return (2020, 1, 0)
    if label == "Feb–Mar 2020":
        return (2020, 1, 1)
    return (int(label[:4]), int(label[-1]), 0)

## Part 0 — SA covariates and continuous spatial metrics

**Goal.** One 2022-SA frame with CBD distance and walkable-stock doses aligned
with the outcome geography. Local time is `dist_to_home < 1 km`, so the **main**
dose is the mean (over January 2020 workers in that SA) of how many PU candidates /
detected TPs / ATM venues lie within 1 km of **that home geohash**. WLS uses
`IHS_Cand`, `IHS_TP`, `IHS_ATM` = `log(1 +` those means `)`. Homes are time-invariant (January 2020),
same worker universe as the outcomes (`profiles` ∩ `work_locations`).

The previous SA-polygon + 500 m halo (`ST_DWITHIN(sa_geom, venue, 500 m)`) is
kept as `IHS_Cand_sa` / `IHS_TP_sa` / `IHS_ATM_sa` for an appendix robustness
check. Do not put both definitions in one horse race. Census 2022 occupation /
industry shares (`it_pct`, `finance_pct`, `academic_occup_pct`,
`academic_degree_pct`) are joined from `phd_census.census_2022` via
`LocalityCode` + `StatArea`.

**Steps.**
1. For each Jan 2020 worker, join `geohash_home` to `phd_adm.geohashes_with_sa_gushdan` (`geohash`, `geometry`) and count PU venues with `ST_DWITHIN(geohash geometry, venue, 1000 m)`; average (and median) to the home SA.
2. Also count venues inside `ST_DWITHIN(sa_geom, 500 m)` (appendix).
3. Buffer area is still computed in SQL (descriptive) but is not a WLS dose.
4. Python maps SQL `n_candidate` / `n_detected` / `n_atm` → `Cand_n` / `TP_n` / `ATM_n`; WLS doses are `IHS_*` = `log1p` and `TP_Ratio = TP_n / (Cand_n + 1)`. Appendix: `IHS_*_sa`. Raw counts stay on `df_sa`.
5. Attach CBS 2022 IT, finance, academic-occupation, and academic-degree
   percentages (time-invariant; absorbed by SA FE unless interacted with `Post`).


In [ ]:
def build_treat_sql(detect_buffer_m, home_disc_m):
    is_atm_only = "COALESCE(CAST(p.flag_atmosphere_suits_rw AS INT64), 0) = 1"
    is_library = type_in_sql("p.primarytype", ("library",))
    is_atm = f"(({is_atm_only}) OR ({is_library}))"
    is_civic = type_in_sql("p.primarytype", CIVIC_CAND_TYPES)
    is_comm = f"NOT ({is_civic})"
    return f"""
WITH cov_ids AS (
  SELECT DISTINCT CAST(yishuv_stat_2022 AS STRING) AS yishuv_stat_2022
  FROM `{COV_TABLE}`
),
sa AS (
  -- Gush Dan SAs only. Simplify first so ST_BUFFER does not exhaust BQ.
  SELECT
    CAST(c.yishuv_stat_2022 AS STRING) AS yishuv_stat_2022,
    c.geometry AS sa_geom,
    ST_AREA(
      ST_BUFFER(ST_SIMPLIFY(c.geometry, 25), {detect_buffer_m}, 2)
    ) / 1e6 AS buffer_area_km2
  FROM `{CENSUS_TABLE}` c
  INNER JOIN cov_ids i
    ON CAST(c.yishuv_stat_2022 AS STRING) = i.yishuv_stat_2022
),
buf AS (
  -- Appendix: PU venues within 500 m of the SA polygon (area + halo).
  SELECT sa.yishuv_stat_2022,
         ANY_VALUE(sa.buffer_area_km2) AS buffer_area_km2,
         COUNT(*) AS n_candidates,
         COUNTIF({is_atm_only}) AS n_candidates_atm,
         COUNTIF({is_atm}) AS n_atm,
         COUNTIF({is_comm}) AS n_candidates_comm,
         COUNTIF({is_civic}) AS n_candidates_noncomm,
         COUNTIF(p.prediction_third_place = 1) AS n_detected
  FROM sa
  JOIN `{PU_TABLE}` p
    ON p.geometry IS NOT NULL
   AND ST_DWITHIN(sa.sa_geom, p.geometry, {detect_buffer_m})
  GROUP BY 1
),
pre_workers AS (
  -- Jan 2020 workers with a known work location (same universe as outcomes).
  SELECT DISTINCT
    CAST(p.identifier AS STRING) AS identifier,
    CAST(p.home_yishuv_stat_2022 AS STRING) AS yishuv_stat_2022,
    w.geohash_home
  FROM `{PROFILES_TABLE}` p
  INNER JOIN `{WORK_TABLE}` w
    ON CAST(w.identifier AS STRING) = CAST(p.identifier AS STRING)
   AND DATE(w.month) = DATE(p.month)
  INNER JOIN cov_ids i
    ON CAST(p.home_yishuv_stat_2022 AS STRING) = i.yishuv_stat_2022
  WHERE DATE(p.month) = DATE '{PRE_MONTH}'
    AND p.home_yishuv_stat_2022 IS NOT NULL
    AND w.geohash_home IS NOT NULL
),
gh_counts AS (
  -- Venues within {home_disc_m} m of each Gush Dan home geohash polygon (spatial index).
  SELECT
    g.geohash,
    COUNT(*) AS n_candidates,
    COUNTIF({is_atm}) AS n_atm,
    COUNTIF(p.prediction_third_place = 1) AS n_detected
  FROM `{GH_TABLE}` g
  JOIN `{PU_TABLE}` p
    ON p.geometry IS NOT NULL
   AND g.geometry IS NOT NULL
   AND ST_DWITHIN(g.geometry, p.geometry, {home_disc_m})
  GROUP BY 1
),
home_counts AS (
  SELECT
    w.identifier,
    w.yishuv_stat_2022,
    COALESCE(c.n_candidates, 0) AS n_candidates,
    COALESCE(c.n_atm, 0) AS n_atm,
    COALESCE(c.n_detected, 0) AS n_detected
  FROM pre_workers w
  INNER JOIN `{GH_TABLE}` g
    ON g.geohash = w.geohash_home
  LEFT JOIN gh_counts c
    ON c.geohash = w.geohash_home
),
home_sa AS (
  SELECT
    yishuv_stat_2022,
    COUNT(*) AS n_homes_dose,
    AVG(n_candidates) AS n_candidate,
    AVG(n_detected) AS n_detected,
    AVG(n_atm) AS n_atm,
    APPROX_QUANTILES(n_candidates, 100)[OFFSET(50)] AS n_candidate_med,
    APPROX_QUANTILES(n_detected, 100)[OFFSET(50)] AS n_detected_med,
    APPROX_QUANTILES(n_atm, 100)[OFFSET(50)] AS n_atm_med
  FROM home_counts
  GROUP BY 1
),
area AS (
  SELECT yishuv_stat_2022, buffer_area_km2 FROM sa
),
census_occ AS (
  SELECT
    CAST(sa.yishuv_stat_2022 AS STRING) AS yishuv_stat_2022,
    c.IndstJ_pcnt AS it_pct,
    c.IndstK_pcnt AS finance_pct,
    c.Occup2_pcnt AS academic_occup_pct,
    c.Acadm1Cert_pcnt AS academic_degree_pct
  FROM `{CENSUS_TABLE}` sa
  LEFT JOIN `{CENSUS_2022_TABLE}` c
    ON c.LocalityCode = sa.semel_yishuv
   AND c.StatArea = sa.stat_2022
)
SELECT
  cov.* EXCEPT (flag_tel_aviv),
  (CAST(cov.semel_yishuv AS INT64) = {TEL_AVIV_SEMEI}) AS flag_tel_aviv,
  COALESCE(home_sa.n_homes_dose, 0) AS n_homes_dose,
  COALESCE(home_sa.n_candidate, 0) AS n_candidate,
  COALESCE(home_sa.n_detected, 0) AS n_detected,
  COALESCE(home_sa.n_atm, 0) AS n_atm,
  home_sa.n_candidate_med,
  home_sa.n_detected_med,
  home_sa.n_atm_med,
  COALESCE(buf.n_candidates, 0) AS n_candidates_sa,
  COALESCE(buf.n_candidates_atm, 0) AS n_candidates_atm,
  COALESCE(buf.n_atm, 0) AS n_atm_sa,
  COALESCE(buf.n_candidates_comm, 0) AS n_candidates_comm,
  COALESCE(buf.n_candidates_noncomm, 0) AS n_candidates_noncomm,
  COALESCE(buf.n_detected, 0) AS n_detected_sa,
  COALESCE(area.buffer_area_km2, 0) AS buffer_area_km2,
  SAFE_DIVIDE(
    COALESCE(buf.n_detected, 0),
    NULLIF(COALESCE(buf.n_candidates, 0), 0)
  ) AS share_detected,
  occ.it_pct,
  occ.finance_pct,
  occ.academic_occup_pct,
  occ.academic_degree_pct
FROM `{COV_TABLE}` cov
LEFT JOIN buf ON buf.yishuv_stat_2022 = cov.yishuv_stat_2022
LEFT JOIN area ON area.yishuv_stat_2022 = cov.yishuv_stat_2022
LEFT JOIN home_sa ON home_sa.yishuv_stat_2022 = cov.yishuv_stat_2022
LEFT JOIN census_occ occ ON occ.yishuv_stat_2022 = cov.yishuv_stat_2022
"""


df_sa = utils.read_query_to_dataframe(build_treat_sql(DETECT_BUFFER_M, OUTBOUND_M))
df_sa["yishuv_stat_2022"] = df_sa["yishuv_stat_2022"].astype(str)
df_sa["flag_tel_aviv"] = (
    pd.to_numeric(df_sa["semel_yishuv"], errors="coerce") == TEL_AVIV_SEMEI
)
df_sa = attach_intensities(df_sa)

print(f"SAs: {len(df_sa):,}")
print(
    f"Jan 2020 homes used for dose: n_homes_dose mean={df_sa.n_homes_dose.mean():.1f}  "
    f"p50={df_sa.n_homes_dose.median():.0f}  SAs with 0 homes="
    f"{int((df_sa.n_homes_dose == 0).sum())}"
)
print("Main (home-1 km mean counts, then log1p WLS doses):")
print(f"  Cand_n mean={df_sa.Cand_n.mean():.3f}  max={df_sa.Cand_n.max():.3f}")
print(f"  TP_n   mean={df_sa.TP_n.mean():.3f}  max={df_sa.TP_n.max():.3f}")
print(f"  ATM_n  mean={df_sa.ATM_n.mean():.3f}  max={df_sa.ATM_n.max():.3f}")
print("Appendix (SA-polygon + 500 m halo counts):")
print(f"  Cand_n_sa mean={df_sa.Cand_n_sa.mean():.3f}  max={df_sa.Cand_n_sa.max():.3f}")
print(f"  TP_n_sa   mean={df_sa.TP_n_sa.mean():.3f}  max={df_sa.TP_n_sa.max():.3f}")
print(f"  ATM_n_sa  mean={df_sa.ATM_n_sa.mean():.3f}  max={df_sa.ATM_n_sa.max():.3f}")
print(
    f"buffer_area_km2 mean={df_sa.buffer_area_km2.mean():.3f}  "
    f"p50={df_sa.buffer_area_km2.median():.3f}  min={df_sa.buffer_area_km2.min():.3f}"
)
print("Main WLS doses (log1p + TP_Ratio):")
print(df_sa[list(DOSE_COLS)].describe().round(3))
print("Home-disc median log1p (optional, not in WLS):")
print(df_sa[["IHS_Cand_med", "IHS_TP_med", "IHS_ATM_med"]].describe().round(3))
print("Corr(home disc, SA-polygon halo):")
print(
    df_sa[list(DOSE_COLS) + list(DOSE_COLS_SA)]
    .corr()
    .loc[list(DOSE_COLS), list(DOSE_COLS_SA)]
    .round(3)
)
print("Census 2022 occupation / industry shares (CBS %):")
print(df_sa[CENSUS_OCC_COLS].describe().round(3))
print("Missing census occ:", df_sa[CENSUS_OCC_COLS].isna().mean().round(3).to_dict())
print("flag_tel_aviv:", df_sa["flag_tel_aviv"].value_counts(dropna=False).to_dict())


In [ ]:
# Diagnostic: raw home-disc means vs log1p WLS doses (Cand / TP / ATM).
fig, axes = plt.subplots(2, 3, figsize=(11.5, 6.2))
raw = [("Cand_n", "Candidates"), ("TP_n", "Detected TPs"), ("ATM_n", "ATM")]
ihs = [("IHS_Cand", "IHS_Cand = log1p(Cand_n)"),
       ("IHS_TP", "IHS_TP = log1p(TP_n)"),
       ("IHS_ATM", "IHS_ATM = log1p(ATM_n)")]
for ax, (col, lab) in zip(axes[0], raw):
    ax.hist(df_sa[col], bins=40, color="#2b6cb0", edgecolor="white")
    ax.set_title(f"Home-disc mean: {lab}")
    ax.set_xlabel(col)
axes[0, 0].set_ylabel("Statistical areas")
for ax, (col, lab) in zip(axes[1], ihs):
    ax.hist(df_sa[col], bins=40, color="#2b6cb0", edgecolor="white")
    ax.set_title(lab)
    ax.set_xlabel(col)
axes[1, 0].set_ylabel("Statistical areas")
fig.tight_layout()
plt.show()

## Step 1 — User-level mobility metrics

**Goal.** Radius of gyration ($r_g$) and local spatial entropy as micro
complements to the SA duration shares.

**$r_g$.** Project dwell points to Israel TM (EPSG:2039). For each user, the
centre of mass is the mean $(x, y)$; $r_g = \sqrt{\mathrm{mean}\{(x-\bar x)^2+(y-\bar y)^2\}}$.
`poi_only=True` drops unnamed dwells.

**Local entropy.** Among named POIs with `dist_to_home <= 1000` m, visit
probabilities $p_i$ yield Shannon entropy $S = -\sum p_i \ln p_i$. One location
$\Rightarrow 0$; no local named POI $\Rightarrow$ NaN.

Functions group by `identifier` only (as specified). They are applied **month by
month** so January 2020 and 2023 months are not pooled inside $r_g$.


In [ ]:
def calculate_radius_of_gyration(dwells_df, poi_only=False):
    """Vectorized r_g (metres) in EPSG:2039. One row per identifier."""
    df = dwells_df
    if poi_only:
        poi = df["poi_id"]
        df = df[poi.notna() & (poi.astype(str) != "") & (poi.astype(str) != "None")]
    if df.empty:
        return pd.DataFrame(columns=["identifier", "rg"])

    gdf = gpd.GeoDataFrame(
        df[["identifier"]].copy(),
        geometry=gpd.points_from_xy(df["lon"].astype(float), df["lat"].astype(float)),
        crs="EPSG:4326",
    )
    xy = gdf.to_crs("EPSG:2039")
    coords = pd.DataFrame(
        {"identifier": xy["identifier"].to_numpy(), "x": xy.geometry.x, "y": xy.geometry.y}
    )
    com = coords.groupby("identifier", sort=False)[["x", "y"]].transform("mean")
    d2 = (coords["x"] - com["x"]) ** 2 + (coords["y"] - com["y"]) ** 2
    rg = np.sqrt(d2.groupby(coords["identifier"], sort=False).mean())
    return rg.rename("rg").reset_index()


def calculate_local_spatial_entropy(dwells_df, radius_m=1000):
    """Shannon entropy of named-POI visit shares within radius_m of home.

    1-location users → 0; 0-location users → NaN.
    """
    ids = dwells_df[["identifier"]].drop_duplicates()
    poi = dwells_df["poi_id"]
    local = dwells_df.loc[
        dwells_df["dist_to_home"].le(radius_m)
        & poi.notna()
        & (poi.astype(str) != "")
        & (poi.astype(str) != "None"),
        ["identifier", "poi_id"],
    ]
    if local.empty:
        return ids.assign(entropy_local=np.nan)

    counts = local.groupby(["identifier", "poi_id"], sort=False).size().rename("n").reset_index()
    tot = counts.groupby("identifier", sort=False)["n"].transform("sum")
    p = counts["n"] / tot
    plogp = p * np.log(p)
    entropy = -plogp.groupby(counts["identifier"], sort=False).sum()
    n_loc = counts.groupby("identifier", sort=False)["poi_id"].nunique()
    entropy = entropy.where(n_loc >= 1, np.nan)
    entropy.loc[n_loc == 1] = 0.0
    out = ids.merge(
        entropy.rename("entropy_local").reset_index(), on="identifier", how="left"
    )
    return out


def user_month_mobility(dwells_df, radius_m=ENTROPY_RADIUS_M):
    """Apply the two functions inside each month; return user × month metrics."""
    frames = []
    for month, part in dwells_df.groupby("month", sort=False):
        rg_all = calculate_radius_of_gyration(part, poi_only=False).rename(columns={"rg": "rg_all"})
        rg_poi = calculate_radius_of_gyration(part, poi_only=True).rename(columns={"rg": "rg_poi"})
        ent = calculate_local_spatial_entropy(part, radius_m=radius_m)
        home = (
            part.groupby("identifier", sort=False)["yishuv_stat_2022"]
            .first()
            .rename("yishuv_stat_2022")
            .reset_index()
        )
        um = (
            rg_all.merge(rg_poi, on="identifier", how="outer")
            .merge(ent, on="identifier", how="outer")
            .merge(home, on="identifier", how="left")
        )
        um["month"] = pd.Timestamp(month)
        frames.append(um)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [ ]:
def build_mobility_dwells_sql(months):
    """Compact lon/lat pull: weekday daytime dwells for workers in `months`."""
    months_list = "', '".join(months)
    h0, h1 = DAYTIME
    return f"""
WITH workers AS (
  SELECT DISTINCT
    CAST(p.identifier AS STRING) AS identifier,
    DATE(p.month) AS month,
    CAST(p.home_yishuv_stat_2022 AS STRING) AS yishuv_stat_2022
  FROM `{PROFILES_TABLE}` p
  INNER JOIN `{WORK_TABLE}` w
    ON CAST(w.identifier AS STRING) = CAST(p.identifier AS STRING)
   AND DATE(w.month) = DATE(p.month)
  WHERE DATE(p.month) IN ('{months_list}')
    AND p.home_yishuv_stat_2022 IS NOT NULL
)
SELECT
  CAST(d.identifier AS STRING) AS identifier,
  DATE(d.month) AS month,
  w.yishuv_stat_2022,
  CAST(d.id AS STRING) AS poi_id,
  d.dist_to_home,
  ST_X(d.geometry) AS lon,
  ST_Y(d.geometry) AS lat
FROM `{DWELLS_TABLE}` d
INNER JOIN workers w
  ON w.identifier = CAST(d.identifier AS STRING)
 AND w.month = DATE(d.month)
WHERE DATE(d.month) IN ('{months_list}')
  AND d.geometry IS NOT NULL
  AND d.duration_hours IS NOT NULL
  AND d.duration_hours > 0
  AND d.start_date_time IS NOT NULL
  AND EXTRACT(DAYOFWEEK FROM DATE(d.start_date_time)) IN (1, 2, 3, 4, 5)
  AND EXTRACT(HOUR FROM d.start_date_time) >= {h0}
  AND EXTRACT(HOUR FROM d.start_date_time) < {h1}
"""


def load_user_mobility(rebuild=REBUILD_MOBILITY):
    if not rebuild and MOBILITY_PARQUET.exists():
        um = pd.read_parquet(MOBILITY_PARQUET)
        um["month"] = pd.to_datetime(um["month"])
        print(f"Loaded mobility parquet: {MOBILITY_PARQUET} ({len(um):,} user-months)")
        return um

    dwells = utils.read_query_to_dataframe(build_mobility_dwells_sql(DID_MONTHS))
    dwells["month"] = pd.to_datetime(dwells["month"])
    dwells["identifier"] = dwells["identifier"].astype(str)
    dwells["yishuv_stat_2022"] = dwells["yishuv_stat_2022"].astype(str)
    dwells = dwells.dropna(subset=["lon", "lat"])
    print(f"Dwells pulled: {len(dwells):,} rows, {dwells['identifier'].nunique():,} users")

    um = user_month_mobility(dwells)
    assert (um["rg_all"].dropna() >= 0).all(), "r_g must be non-negative"
    assert um["entropy_local"].dropna().ge(0).all(), "entropy must be >= 0 or NaN"
    n_one = (um["entropy_local"] == 0).sum()
    n_nan = um["entropy_local"].isna().sum()
    print(f"entropy: {n_one:,} one-location (0); {n_nan:,} zero-location (NaN)")

    MOBILITY_PARQUET.parent.mkdir(parents=True, exist_ok=True)
    um.to_parquet(MOBILITY_PARQUET, index=False)
    print(f"Wrote {MOBILITY_PARQUET}")
    try:
        utils.upload_table_to_bq(
            um.drop(columns=["geometry"], errors="ignore"),
            "phd_datasets",
            "sa_retention_user_mobility_3302",
            geometry_column=None,
            if_table_exist="replace",
        )
    except Exception as exc:
        print(f"BQ mobility upload skipped: {exc}")
    return um


df_user_mob = load_user_mobility()

df_mob_sa = (
    df_user_mob.groupby(["yishuv_stat_2022", "month"], as_index=False)
    .agg(
        rg_mean=("rg_all", "mean"),
        rg_poi_mean=("rg_poi", "mean"),
        entropy_local_mean=("entropy_local", "mean"),
        n_users_mob=("identifier", "nunique"),
    )
)
print(f"SA-month mobility rows: {len(df_mob_sa):,}")
print(df_mob_sa[["rg_mean", "rg_poi_mean", "entropy_local_mean"]].describe().round(2))

## Step 2 — Continuous share time-series (Jan 2020 – 2023Q3)

**Goal.** A visual narrative of the eight duration shares over time, aggregated
to calendar quarters so small-sample months do not dominate. **January 2020 is
its own tick** and is not pooled with February–March 2020 (the rest of 2020Q1).
`share_poi_of_local_h` (named local POI hours / local weekday-daytime hours) is panel 6 of
Figs. 1–2b. Radius of gyration (`rg_mean`, `rg_poi_mean`) is Fig. 1b: one
figure, two panels, six Dist × IHS-TP series on each, on whatever SA-months
mobility has been pulled (DiD months unless `load_user_mobility` is rebuilt
on `QUARTER_MONTHS`).

**2019Q3 is omitted** from Figs. 1–2b: only one SA in July 2019 clears
`MIN_WORKERS` / `MIN_WORKER_DAYS`; August–December 2019 have none.

These shares are **not mutually exclusive** — each has its own denominator — so
they are plotted as separate lines, not a stacked composition.

**Mapping (Fig. 1–2b shares; Fig. 1b gyration).**

```text
Metric: each duration share | rg_mean / rg_poi_mean (m)
Categories (shares): all SAs; IHS_TP tertile; Near/Far CBD
Categories (gyration): dist_split × tp_tertile (6 cells)
X: quarter (Jan 2020 separate); 95% normal CI on worker-weighted SA mean
```


In [ ]:
def build_outcomes_sql(outbound_m, outcome_table, months):
    """Daytime weekday SA-month duration shares (3300 allocation, extra 3302 metrics)."""
    months_list = "', '".join(months)
    did_months_list = "', '".join(DID_MONTHS)
    n_did_months = len(DID_MONTHS)
    h0, h1 = DAYTIME
    min_h = round(MIN_WINDOW_COVERAGE * (h1 - h0), 4)
    return f"""
CREATE OR REPLACE TABLE `{outcome_table}` AS
WITH workers AS (
  SELECT DISTINCT
    CAST(p.identifier AS STRING) AS identifier,
    DATE(p.month) AS month,
    CAST(p.home_yishuv_stat_2022 AS STRING) AS yishuv_stat_2022
  FROM `{PROFILES_TABLE}` p
  INNER JOIN `{WORK_TABLE}` w
    ON CAST(w.identifier AS STRING) = CAST(p.identifier AS STRING)
   AND DATE(w.month) = DATE(p.month)
  WHERE DATE(p.month) IN ('{months_list}')
    AND p.home_yishuv_stat_2022 IS NOT NULL
),
dwells AS (
  SELECT
    CAST(d.identifier AS STRING) AS identifier,
    DATE(d.month) AS month,
    DATETIME(d.start_date_time) AS local_start,
    DATETIME_ADD(
      DATETIME(d.start_date_time),
      INTERVAL CAST(ROUND(d.duration_hours * 3600) AS INT64) SECOND
    ) AS local_end,
    COALESCE(d.flag_home, FALSE) AS flag_home,
    COALESCE(d.flag_work, FALSE) AS flag_work,
    d.dist_to_home,
    CAST(d.id AS STRING) AS poi_id,
    w.yishuv_stat_2022,
    (pu.id IS NOT NULL) AS is_candidate,
    COALESCE(pu.is_detected, FALSE) AS is_detected
  FROM `{DWELLS_TABLE}` d
  INNER JOIN workers w
    ON w.identifier = CAST(d.identifier AS STRING)
   AND w.month = DATE(d.month)
  LEFT JOIN (
    SELECT
      CAST(id AS STRING) AS id,
      LOGICAL_OR(prediction_third_place = 1) AS is_detected
    FROM `{PU_TABLE}`
    WHERE id IS NOT NULL
    GROUP BY 1
  ) pu ON pu.id = CAST(d.id AS STRING)
  WHERE DATE(d.month) IN ('{months_list}')
    AND d.duration_hours IS NOT NULL
    AND d.duration_hours > 0
    AND d.start_date_time IS NOT NULL
),
poi_ok AS (
  -- Named POIs with ≥ MIN_POI_DWELLS_PER_MONTH worker dwells in every DiD month.
  SELECT poi_id
  FROM (
    SELECT poi_id, month, COUNT(*) AS n_dwells
    FROM dwells
    WHERE poi_id IS NOT NULL AND poi_id != ''
      AND month IN ('{did_months_list}')
    GROUP BY poi_id, month
  )
  GROUP BY poi_id
  HAVING COUNTIF(n_dwells >= {MIN_POI_DWELLS_PER_MONTH}) = {n_did_months}
),
alloc AS (
  SELECT
    d.yishuv_stat_2022,
    d.month,
    d.identifier,
    dd AS local_date,
    GREATEST(0, DATETIME_DIFF(
      LEAST(d.local_end, DATETIME_ADD(DATETIME(dd), INTERVAL {h1} HOUR)),
      GREATEST(d.local_start, DATETIME_ADD(DATETIME(dd), INTERVAL {h0} HOUR)),
      SECOND)) / 3600.0 AS dur_h,
    (d.poi_id IS NOT NULL AND d.poi_id != '' AND ok.poi_id IS NOT NULL) AS is_poi,
    (d.dist_to_home IS NOT NULL AND d.dist_to_home < {outbound_m}) AS is_local,
    COALESCE(d.flag_home, FALSE) AS is_home,
    COALESCE(d.flag_work, FALSE) AS is_work,
    (NOT COALESCE(d.flag_home, FALSE)) AS is_nothome,
    (NOT COALESCE(d.flag_home, FALSE) AND NOT COALESCE(d.flag_work, FALSE)) AS is_not_hw,
    (COALESCE(d.is_candidate, FALSE) AND ok.poi_id IS NOT NULL) AS is_candidate,
    (COALESCE(d.is_detected, FALSE) AND ok.poi_id IS NOT NULL) AS is_detected
  FROM dwells d
  LEFT JOIN poi_ok ok ON ok.poi_id = d.poi_id
  CROSS JOIN UNNEST([DATE(d.local_start), DATE_ADD(DATE(d.local_start), INTERVAL 1 DAY)]) AS dd
),
alloc_wd AS (
  SELECT * FROM alloc
  WHERE dur_h > 0
    AND EXTRACT(DAYOFWEEK FROM local_date) IN (1, 2, 3, 4, 5)
),
user_day AS (
  SELECT yishuv_stat_2022, month, identifier, local_date, SUM(dur_h) AS dur_total
  FROM alloc_wd
  GROUP BY 1, 2, 3, 4
),
user_day_ok AS (
  SELECT * FROM user_day WHERE dur_total >= {min_h}
),
alloc_ok AS (
  SELECT a.*
  FROM alloc_wd a
  JOIN user_day_ok u
    USING (yishuv_stat_2022, month, identifier, local_date)
)
SELECT
  yishuv_stat_2022,
  month,
  COUNT(DISTINCT identifier) AS n_workers,
  COUNT(DISTINCT CONCAT(identifier, '|', CAST(local_date AS STRING))) AS n_worker_days,
  SUM(dur_h) AS dur_total,
  SUM(IF(is_local, dur_h, 0)) AS dur_local_all_h,
  SAFE_DIVIDE(SUM(IF(is_local, dur_h, 0)), SUM(dur_h)) AS share_local_all_h,
  SAFE_DIVIDE(
    SUM(IF(is_home AND NOT is_work, dur_h, 0)),
    SUM(IF(NOT is_work, dur_h, 0))
  ) AS share_home_nonwork_h,
  SAFE_DIVIDE(
    SUM(IF(is_local AND is_not_hw, dur_h, 0)),
    SUM(IF(is_not_hw, dur_h, 0))
  ) AS share_local_nothome_nonwork_h,
  SAFE_DIVIDE(SUM(IF(is_poi, dur_h, 0)), SUM(dur_h)) AS share_poi_all_h,
  SAFE_DIVIDE(SUM(IF(is_local AND is_poi, dur_h, 0)), SUM(IF(is_poi, dur_h, 0))) AS share_local_poi_h,
  -- Named local POI / local hours. dur_h is already weekday daytime overlap
  -- (Sun–Thu, 06:00–22:00) from alloc_wd; nights and weekends are not in either side.
  SAFE_DIVIDE(
    SUM(IF(is_local AND is_poi, dur_h, 0)),
    SUM(IF(is_local, dur_h, 0))
  ) AS share_poi_of_local_h,
  SAFE_DIVIDE(
    SUM(IF(is_local AND is_candidate, dur_h, 0)),
    SUM(IF(is_candidate, dur_h, 0))
  ) AS share_local_cand_h,
  SAFE_DIVIDE(SUM(IF(is_detected, dur_h, 0)), SUM(IF(is_poi, dur_h, 0))) AS share_tp_poi_h,
  -- Flag consistency (should be 0): detected hours outside the candidate set.
  SUM(IF(is_detected AND NOT is_candidate, dur_h, 0)) AS dur_detected_not_cand
FROM alloc_ok
GROUP BY 1, 2
HAVING COUNT(DISTINCT identifier) >= {MIN_WORKERS}
   AND COUNT(DISTINCT CONCAT(identifier, '|', CAST(local_date AS STRING))) >= {MIN_WORKER_DAYS}
"""


def load_quarterly_outcomes(rebuild=REBUILD_QUARTERLY):
    if rebuild:
        utils.read_query_to_dataframe(
            build_outcomes_sql(OUTBOUND_M, OUTCOME_TABLE_3302, QUARTER_MONTHS)
        )
    win = utils.read_query_to_dataframe(f"SELECT * FROM `{OUTCOME_TABLE_3302}`")
    win["month"] = pd.to_datetime(win["month"])
    win["yishuv_stat_2022"] = win["yishuv_stat_2022"].astype(str)
    return win.sort_values(["yishuv_stat_2022", "month"], ignore_index=True)


df_win = load_quarterly_outcomes()
df_win = attach_local_volume(df_win)

for c in SHARE_OUTCOMES:
    s = df_win[c].dropna()
    assert s.between(0, 1).all(), f"{c} outside [0, 1]"
assert (df_win["dur_detected_not_cand"].fillna(0) == 0).all(), (
    "is_detected should imply is_candidate (eligible named POI)"
)

print(f"SA-months: {len(df_win):,}  SAs: {df_win['yishuv_stat_2022'].nunique():,}")
print("SAs by month:")
print(df_win.groupby("month")["yishuv_stat_2022"].nunique().head(8))
print(df_win[SHARE_OUTCOMES].describe().round(3))


In [ ]:
# Attach time-invariant SA dose.
# Low = no detected TPs (IHS_TP = 0).
# Mid = positive IHS_TP up to the 75th percentile of *non-zero* values.
# High = the remainder (above that p75). Same cut as 3300 tp_bin 1 vs 2.
ihs_nz = df_sa.loc[df_sa["IHS_TP"] > 0, "IHS_TP"]
IHS_TP_P75 = float(ihs_nz.quantile(0.75))
df_sa["tp_tertile"] = np.where(
    df_sa["IHS_TP"] <= 0,
    "Low IHS-TP",
    np.where(df_sa["IHS_TP"] <= IHS_TP_P75, "Mid IHS-TP", "High IHS-TP"),
)
print(
    f"Non-zero IHS_TP p75 = {IHS_TP_P75:.4f}  "
    f"(TP_n equivalent = {np.expm1(IHS_TP_P75):.4f} mean detected TPs in the home 1 km disc)"
)
print(df_sa["tp_tertile"].value_counts().rename("n_SA"))
print(df_sa.groupby("tp_tertile")["IHS_TP"].agg(["min", "median", "max"]))
df_sa["dist_split"] = np.where(
    df_sa["Dist"] <= df_sa["Dist"].median(), "Near CBD", "Far CBD"
)

q = df_win.merge(
    df_sa[["yishuv_stat_2022", "IHS_TP", "TP_n", "tp_tertile", "Dist", "dist_split"]],
    on="yishuv_stat_2022",
    how="left",
)
mob_m = df_mob_sa.copy()
mob_m["month"] = pd.to_datetime(mob_m["month"])
mob_m["yishuv_stat_2022"] = mob_m["yishuv_stat_2022"].astype(str)
q = q.merge(mob_m, on=["yishuv_stat_2022", "month"], how="left")
q["year_quarter"] = q["month"].dt.to_period("Q").astype(str)
q["period"] = q["month"].map(to_plot_period)

PLOT_MEAN_COLS = SHARE_OUTCOMES + GYRATION_OUTCOMES

# SA × calendar-quarter means (then 2-quarter rolling per SA).
# 2020Q1 here still pools Jan–Mar; the share plots use `period` instead.
sa_q = (
    q.groupby(["yishuv_stat_2022", "year_quarter", "tp_tertile", "dist_split"], observed=True)
    .agg(
        **{c: (c, "mean") for c in PLOT_MEAN_COLS},
        n_workers=("n_workers", "sum"),
        month=("month", "min"),
    )
    .reset_index()
    .sort_values(["yishuv_stat_2022", "year_quarter"])
)
for c in SHARE_OUTCOMES:
    sa_q[f"{c}_roll2"] = (
        sa_q.groupby("yishuv_stat_2022")[c].transform(lambda s: s.rolling(2, min_periods=1).mean())
    )

hy = (
    q.groupby(["yishuv_stat_2022", "period", "tp_tertile", "dist_split"], observed=True)
    .agg(
        **{c: (c, "mean") for c in PLOT_MEAN_COLS},
        n_workers=("n_workers", "sum"),
    )
    .reset_index()
)
print(f"SA-quarters: {len(sa_q):,}  SA-plot-periods: {len(hy):,}")
print("Plot periods:", sorted(hy["period"].unique(), key=period_sort_key))

n_sa_2019q3 = int(hy.loc[hy["period"] == "2019Q3", "yishuv_stat_2022"].nunique())
print(
    f"Note: 2019Q3 omitted from Figs. 1–2b — {n_sa_2019q3} SA "
    "(July 2019 only; does not clear a usable panel). "
    "2019Q4 has no SA-months above MIN_WORKERS / MIN_WORKER_DAYS."
)
hy = hy[hy["period"] != "2019Q3"].copy()
print("Plot periods after drop:", sorted(hy["period"].unique(), key=period_sort_key))


In [ ]:
PALETTE_TP = {
    "Low IHS-TP": "#718096",
    "Mid IHS-TP": "#3182ce",
    "High IHS-TP": "#c05621",
}


def plot_share_panels(
    plot_df,
    series_col,
    series_order,
    palette,
    title,
    fname,
    metrics=None,
    labels=None,
    ylabel="Share",
    share_ylim=True,
):
    """One panel per metric; lines + 95% normal CI on weighted SA means.

    Default metrics = SHARE_OUTCOMES (includes share_poi_of_local_h).
    """
    metrics = list(SHARE_OUTCOMES if metrics is None else metrics)
    labels = {**SHARE_LABELS, **MOBILITY_LABELS} if labels is None else labels
    missing = [m for m in metrics if m not in plot_df.columns]
    if missing:
        raise KeyError(
            f"plot_df missing {missing}. Re-run config, then the Step 2 aggregation cell."
        )
    print("Panels:", [f"{labels.get(m, m)}  ({m})" for m in metrics])
    n_m = len(metrics)
    ncols = 2 if n_m >= 6 else 1
    nrows = int(np.ceil(n_m / ncols))
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(5.6 * ncols + 1.2, 2.35 * nrows),
        sharex=True,
    )
    axes = np.atleast_1d(axes).ravel()
    x_order = sorted(plot_df["period"].unique(), key=period_sort_key)

    for ax, metric in zip(axes, metrics):
        agg = weighted_mean_ci(
            plot_df, ["period", series_col], metric, weight_col="n_workers"
        )
        for i, series in enumerate(series_order):
            d = (
                agg[agg[series_col] == series]
                .set_index("period")
                .reindex(x_order)
                .reset_index()
            )
            x = np.arange(len(x_order))
            color = palette[i] if not isinstance(palette, dict) else palette[series]
            ax.plot(x, d["mean"], marker="o", ms=4.5, color=color, label=series, lw=1.6)
            ax.fill_between(x, d["ci_low"], d["ci_high"], color=color, alpha=0.18)
        ax.set_ylabel(ylabel)
        ax.set_title(labels[metric], loc="left", fontsize=11)
        if share_ylim:
            ymin = max(0.0, float(np.nanmin(agg["ci_low"])) - 0.02)
            ymax = min(1.0, float(np.nanmax(agg["ci_high"])) + 0.02)
            if ymax - ymin < 0.08:
                mid = 0.5 * (ymin + ymax)
                ymin, ymax = mid - 0.05, mid + 0.05
        else:
            ymin = float(np.nanmin(agg["ci_low"]))
            ymax = float(np.nanmax(agg["ci_high"]))
            pad = 0.05 * (ymax - ymin if ymax > ymin else 1.0)
            ymin, ymax = ymin - pad, ymax + pad
        ax.set_ylim(ymin, ymax)
        ax.legend(
            fontsize=7 if len(series_order) >= 5 else 8,
            loc="best",
            frameon=False,
            ncol=2 if len(series_order) >= 5 else 1,
        )
        ax.set_xticks(np.arange(len(x_order)))
        ax.set_xticklabels(x_order, rotation=30, ha="right")

    for ax in axes[n_m:]:
        ax.set_visible(False)
    for ax in axes[max(0, n_m - ncols):n_m]:
        ax.set_xlabel("Quarter (Jan 2020 separate from Feb–Mar)")
    fig.suptitle(title, y=1.01, fontsize=12)
    fig.tight_layout()
    out = _nb_dir / fname
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")


hy["series_all"] = "All SAs"
plot_share_panels(
    hy,
    series_col="series_all",
    series_order=["All SAs"],
    palette=["#2b6cb0"],
    title="Weekday daytime duration shares, Gush Dan home SAs",
    fname="3302_fig1_shares_quarter.png",
    metrics=SHARE_OUTCOMES,
)

### ONLY FOR SA WITH 50%+ IT / FIN / ACADEMIC OCCUPATIONS

In [ ]:
OCC_SHARE_MIN = 50.0
occ_cols = ["it_pct", "finance_pct", "academic_occup_pct"]
sa_occ = df_sa[["yishuv_stat_2022", *occ_cols]].copy()
for c in occ_cols:
    sa_occ[c] = pd.to_numeric(sa_occ[c], errors="coerce")
sa_occ["occ_it_fin_acad"] = sa_occ[occ_cols].sum(axis=1, min_count=3)
keep_sa = set(
    sa_occ.loc[sa_occ["occ_it_fin_acad"] > OCC_SHARE_MIN, "yishuv_stat_2022"].astype(str)
)
hy_occ = hy[hy["yishuv_stat_2022"].isin(keep_sa)].copy()
hy_occ["series_all"] = "All SAs"

In [ ]:
plot_share_panels(
    hy_occ,
    series_col="series_all",
    series_order=["All SAs"],
    palette=["#2b6cb0"],
    title="Weekday daytime duration shares, Gush Dan home SAs",
    fname="3302_fig1_shares_quarter.png",
    metrics=SHARE_OUTCOMES,
)

In [ ]:
plot_share_panels(
    hy.dropna(subset=["tp_tertile"]),
    series_col="tp_tertile",
    series_order=["Low IHS-TP", "Mid IHS-TP", "High IHS-TP"],
    palette=PALETTE_TP,
    title="Duration shares by home-SA third-place intensity (zero / p75 / tail)",
    fname="3302_fig2_shares_by_ihs_tp.png",
    metrics=SHARE_OUTCOMES,
)

In [ ]:
PALETTE_DIST = {"Near CBD": "#2b6cb0", "Far CBD": "#c05621"}
plot_share_panels(
    hy.dropna(subset=["dist_split"]),
    series_col="dist_split",
    series_order=["Near CBD", "Far CBD"],
    palette=PALETTE_DIST,
    title="Duration shares by distance to CBD (median split)",
    fname="3302_fig2b_shares_by_dist.png",
    metrics=SHARE_OUTCOMES,
)

In [ ]:
# Metric: rg_mean / rg_poi_mean (m). Categories: dist_split × tp_tertile (6). X: metres.
hy_rg = hy.dropna(subset=GYRATION_OUTCOMES, how="all").copy()
hy_rg = hy_rg.dropna(subset=["dist_split", "tp_tertile"])
hy_rg["dist_tp"] = (
    hy_rg["dist_split"].astype(str) + " × " + hy_rg["tp_tertile"].astype(str)
)

order_6 = [
    "Near CBD × Low IHS-TP",
    "Near CBD × Mid IHS-TP",
    "Near CBD × High IHS-TP",
    "Far CBD × Low IHS-TP",
    "Far CBD × Mid IHS-TP",
    "Far CBD × High IHS-TP",
]
PALETTE_DIST_TP = {
    "Near CBD × Low IHS-TP": "#90cdf4",
    "Near CBD × Mid IHS-TP": "#3182ce",
    "Near CBD × High IHS-TP": "#2c5282",
    "Far CBD × Low IHS-TP": "#fbd38d",
    "Far CBD × Mid IHS-TP": "#dd6b20",
    "Far CBD × High IHS-TP": "#9c4221",
}

print(
    "Gyration plot periods: "
    f"{sorted(hy_rg['period'].unique(), key=period_sort_key)}  "
    f"(n SA-periods with rg = {hy_rg['rg_mean'].notna().sum():,})"
)
print(hy_rg.drop_duplicates("yishuv_stat_2022")["dist_tp"].value_counts())

if hy_rg.empty:
    print("SKIP gyration plots: merge df_mob_sa onto q (re-run Step 2 aggregation).")
else:
    plot_share_panels(
        hy_rg,
        series_col="dist_tp",
        series_order=order_6,
        palette=PALETTE_DIST_TP,
        title="Radius of gyration by CBD distance × third-place intensity",
        fname="3302_fig1b_gyration_dist_x_tp.png",
        metrics=GYRATION_OUTCOMES,
        ylabel="Metres",
        share_ylim=False,
    )
    print(
        "Fig. 1b. Worker-weighted mean radius of gyration by quarter, "
        "six Dist × IHS-TP cells on each panel. Top: all daytime dwells "
        "(home, work, unnamed). Bottom: named POIs only. Near CBD = blues, "
        "Far CBD = oranges; darker = higher IHS-TP (Low = 0 detected TPs; "
        "Mid = positive up to p75 of non-zero; High = tail). "
        "Shaded bands are 95% normal CIs on the SA mean (weights = n_workers). "
        "Notes: Mobility is currently pulled for DiD months (Jan 2020, 2023Q1) "
        "unless load_user_mobility is rebuilt on QUARTER_MONTHS. Same sample as "
        "Fig. 1 where months overlap."
    )


## Step 3 — DiD panel (pooled pre / pooled post)

Isolate January 2020 and **2023Q1** (January–March 2023). Keep month grain (four
time points) so `C(year_quarter)` can enter as time FE. `Post = 1` only when
`year_quarter == 2023Q1` — later 2023 months in the descriptive series are not
post. `Post` is collinear with `C(year_quarter)` on this two-quarter panel; the
triple interaction is still identified.

WLS weights are the January 2020 worker count (`n_workers_pre`), held constant
within SA. Mobility means are merged from Step 1. Home-disc log1p doses (`IHS_Cand`,
`IHS_TP`, `IHS_ATM`) and appendix polygon-halo doses (`IHS_*_sa`) are merged from
Part 0. Census 2022 occupation / industry shares (`it_pct`, `finance_pct`,
`academic_occup_pct`, `academic_degree_pct`) come from Part 0; they are
time-invariant and absorbed by SA FE unless interacted with `Post`.


In [ ]:
PRE_TS = pd.Timestamp(PRE_MONTH)
POST_TS = [pd.Timestamp(x) for x in POST_MONTHS]
DID_TS = [PRE_TS, *POST_TS]

did = df_win[df_win["month"].isin(DID_TS)].copy()
pre_ids = set(did.loc[did["month"] == PRE_TS, "yishuv_stat_2022"])
post_ids = set(did.loc[did["month"].isin(POST_TS), "yishuv_stat_2022"])
keep = pre_ids & post_ids
did = did[did["yishuv_stat_2022"].isin(keep)].copy()

n_pre = (
    did.loc[did["month"] == PRE_TS, ["yishuv_stat_2022", "n_workers"]]
    .drop_duplicates("yishuv_stat_2022")
    .rename(columns={"n_workers": "n_workers_pre"})
)
did = did.merge(n_pre, on="yishuv_stat_2022", how="inner")
_sa_cols = list(dict.fromkeys([
    "yishuv_stat_2022",
    "Dist",
    *DOSE_COLS,
    *DOSE_COLS_SA,
    "buffer_area_km2",
    "semel_yishuv",
    "flag_tel_aviv",
    *CENSUS_OCC_COLS,
]))
did = did.merge(df_sa[_sa_cols], on="yishuv_stat_2022", how="inner")
did = did.drop(columns=["Cand_n", "TP_n", "ATM_n", "Cand_dens", "TP_dens", "ATM_dens"], errors="ignore")
did = did.merge(df_mob_sa, on=["yishuv_stat_2022", "month"], how="left")

did["year_quarter"] = did["month"].dt.to_period("Q").astype(str).astype("object")
did["Post"] = (did["year_quarter"] == "2023Q1").astype(int)
did["yishuv_stat_2022"] = did["yishuv_stat_2022"].astype(str).astype("object")
did = attach_local_volume(did)
did["log_dist"] = np.log1p(pd.to_numeric(did["Dist"], errors="coerce"))  # log(Dist + 1 km)
for c in CENSUS_OCC_COLS:
    did[c] = pd.to_numeric(did[c], errors="coerce").fillna(0.0)
assert did.loc[did["year_quarter"] != "2023Q1", "Post"].eq(0).all()
assert did.loc[did["year_quarter"] == "2023Q1", "Post"].eq(1).all()

print(
    f"Panel: {len(did):,} SA-months, {did['yishuv_stat_2022'].nunique():,} SAs "
    f"(pre ∩ post). year_quarter = {sorted(did['year_quarter'].unique())}"
)
print(did.groupby(["year_quarter", "Post"]).size().rename("n"))
print("n_workers_pre:", did.drop_duplicates("yishuv_stat_2022")["n_workers_pre"].describe().round(1))
print("Missing rg_mean:", did["rg_mean"].isna().mean())
print("Missing rg_poi_mean:", did["rg_poi_mean"].isna().mean())
print("Missing entropy_local_mean:", did["entropy_local_mean"].isna().mean())
print("Localities (semel_yishuv):", did["semel_yishuv"].nunique())
print("flag_tel_aviv SAs:", did.drop_duplicates("yishuv_stat_2022")["flag_tel_aviv"].value_counts().to_dict())
print("Home-disc vs polygon-halo doses (one row per SA):")
print(
    did.drop_duplicates("yishuv_stat_2022")[list(DOSE_COLS) + list(DOSE_COLS_SA)]
    .describe()
    .round(3)
)
print("Missing census occ (panel):", did[CENSUS_OCC_COLS].isna().mean().round(3).to_dict())
print(
    did.drop_duplicates("yishuv_stat_2022")[CENSUS_OCC_COLS]
    .describe()
    .round(3)
)
print(
    did[["dur_local_all_h", "local_h_per_worker", "log_local_h_per_worker", "log_dist"]]
    .describe()
    .round(3)
)

## Step 3b — Can `Post × Dist` be split from `Post × Tel Aviv`?

`Post × flag_tel_aviv` (holding Dist fixed) is identified only where Tel Aviv
and non–Tel Aviv statistical areas sit at **similar kilometres** from Azrieli
(e.g. north TA vs Ramat Gan / Giv'atayim). If the Dist supports barely overlap,
the two `Post` interactions are collinear.

```text
Metric: Dist (km from home SA centroid to Azrieli)
Categories: flag_tel_aviv (Tel Aviv vs rest of Gush Dan)
X: Dist (km); mean Dist with 95% normal CI
```

Unit of analysis = one SA in the Step 3 DiD sample (pre ∩ post).


In [ ]:
# Metric: Dist (km). Categories: flag_tel_aviv. X: Dist (km) / mean Dist.
sa_did = (
    did.drop_duplicates("yishuv_stat_2022")[["yishuv_stat_2022", "Dist", "flag_tel_aviv"]]
    .dropna(subset=["Dist", "flag_tel_aviv"])
    .copy()
)
sa_did["flag_tel_aviv"] = sa_did["flag_tel_aviv"].astype(bool)
sa_did["ta_label"] = np.where(
    sa_did["flag_tel_aviv"], "Tel Aviv (semel 5000)", "Rest of Gush Dan"
)
PALETTE_TA = {"Tel Aviv (semel 5000)": "#c05621", "Rest of Gush Dan": "#2b6cb0"}
order_ta = ["Rest of Gush Dan", "Tel Aviv (semel 5000)"]

dist_by = (
    sa_did.groupby("ta_label", observed=True)["Dist"]
    .agg(mean="mean", sd="std", n="size", min="min", p25=lambda s: s.quantile(0.25),
         median="median", p75=lambda s: s.quantile(0.75), max="max")
    .reset_index()
)
se = dist_by["sd"] / np.sqrt(dist_by["n"])
dist_by = dist_by.assign(ci_low=dist_by["mean"] - 1.96 * se, ci_high=dist_by["mean"] + 1.96 * se)
print("Mean Dist (km) by municipality group, 95% normal CI")
print(dist_by.set_index("ta_label").loc[order_ta].round(3).to_string())
print()
print(
    "corr(Dist, flag_tel_aviv) = "
    f"{sa_did['Dist'].corr(sa_did['flag_tel_aviv'].astype(int)):.3f}  "
    f"(n = {len(sa_did):,} SAs)"
)

ta = sa_did.loc[sa_did["flag_tel_aviv"], "Dist"]
rest = sa_did.loc[~sa_did["flag_tel_aviv"], "Dist"]
lo, hi = max(ta.min(), rest.min()), min(ta.max(), rest.max())
in_cs = sa_did["Dist"].between(lo, hi)
print(
    f"Common support on Dist: [{lo:.2f}, {hi:.2f}] km  |  "
    f"SAs in overlap: TA {int((sa_did['flag_tel_aviv'] & in_cs).sum())}  "
    f"rest {int((~sa_did['flag_tel_aviv'] & in_cs).sum())}"
)

BIN_KM = 2.0
sa_did["dist_bin"] = (np.floor(sa_did["Dist"] / BIN_KM) * BIN_KM).astype(float)
bin_n = (
    sa_did.groupby(["dist_bin", "ta_label"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=order_ta, fill_value=0)
    .sort_index()
)
both = (bin_n[order_ta[0]] > 0) & (bin_n[order_ta[1]] > 0)
print(f"\nSAs per {BIN_KM:.0f} km Dist bin (bins with both groups identify Post:TA | Dist):")
print(bin_n.assign(both_groups=both).to_string())
print(f"Bins with both groups: {int(both.sum())} / {len(bin_n)}")

fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.0),
                         gridspec_kw={"width_ratios": [1.15, 0.85, 1.35]})

ax = axes[0]
bins = np.arange(0, sa_did["Dist"].max() + BIN_KM, BIN_KM)
for lab in order_ta:
    vals = sa_did.loc[sa_did["ta_label"] == lab, "Dist"]
    ax.hist(
        vals, bins=bins, density=True, alpha=0.45,
        color=PALETTE_TA[lab], edgecolor="white",
        label=f"{lab} (n={len(vals)})",
    )
    ax.axvline(vals.mean(), color=PALETTE_TA[lab], ls="--", lw=1.2)
ax.set_xlabel("Distance to CBD (km)")
ax.set_ylabel("Density")
ax.set_title("Dist density by Tel Aviv status")
ax.legend(fontsize=7.5, loc="upper right")

ax = axes[1]
y = np.arange(len(order_ta))
tab = dist_by.set_index("ta_label").loc[order_ta]
ax.errorbar(
    tab["mean"], y,
    xerr=[tab["mean"] - tab["ci_low"], tab["ci_high"] - tab["mean"]],
    fmt="o", ms=7, color="#2b6cb0", ecolor="#2b6cb0", elinewidth=1.6, capsize=3,
)
ax.set_yticks(y)
ax.set_yticklabels(order_ta)
ax.set_ylim(len(order_ta) - 0.5, -0.5)
ax.set_xlabel("Mean Dist (km)")
ax.set_title("Mean Dist, 95% normal CI")
ax.axvline(sa_did["Dist"].mean(), ls=":", lw=1.1, color="#718096",
           label=f"All SAs ({sa_did['Dist'].mean():.1f} km)")
ax.legend(fontsize=7.5, loc="best")

ax = axes[2]
x = bin_n.index.to_numpy() + BIN_KM / 2
width = BIN_KM * 0.38
for i, lab in enumerate(order_ta):
    ax.bar(
        x + (i - 0.5) * width, bin_n[lab].to_numpy(),
        width=width, color=PALETTE_TA[lab], label=lab, edgecolor="white",
    )
ax.set_xlabel("Distance to CBD (km)")
ax.set_ylabel("Number of SAs")
ax.set_title(f"SAs per {BIN_KM:.0f} km bin (common support)")
ax.legend(fontsize=7.5, loc="upper right")

fig.suptitle("Home-SA distance to Azrieli, Tel Aviv vs rest of Gush Dan", y=1.02, fontsize=12)
fig.tight_layout()
out = _nb_dir / "3302_fig2c_dist_by_telaviv.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")
print(
    "Fig. 2c. Home statistical-area distance to the CBD (Azrieli) in the DiD "
    "sample, Tel Aviv municipality (CBS locality 5000) vs the rest of Gush Dan. "
    "Left: overlapping Dist densities (2 km bins). Centre: mean Dist with 95% "
    "normal CIs. Right: SA counts by 2 km Dist bin; Post × Tel Aviv holding Dist "
    f"fixed is identified in bins with both groups ({int(both.sum())} of {len(bin_n)}). "
    f"Common support is {lo:.1f}–{hi:.1f} km. Notes: One SA per row; pre ∩ post "
    "panel from Step 3."
)


## Step 4 — Spatial, demographic, and amenity WLS models

Five nested specifications on the Step 3 panel. For every equation, $Y_{it}$ is a
continuous duration share (or an SA mobility mean), $\alpha_i$ absorbs SA
unobservables (`yishuv_stat_2022`), and $\gamma_{q(t)}$ absorbs year-quarter
macro-shocks. Dist in the formula is $\ln(1+\mathrm{Dist}_i)$. WLS weights =
`n_workers_pre`; SEs clustered at SA.

**Core spatial & demographic.** Model 1 (donut) then Model 2 (WFH capacity).
**Amenity & integration.** Model 3 (amenity pull), Model 4 (additive horse race),
Model 5 (WFH × dose synergy, no Dist). Amenity doses are home-1 km means. Polygon-halo
doses are a separate appendix; they are not entered in the same equation.

### Model 1 — Baseline spatial shift (pure donut)

Macro-geography of the pandemic shock: whether the friction of distance alone
drove decentralisation.

$$
Y_{it} = \alpha_i + \gamma_{q(t)} + \beta_1\bigl(\mathrm{Post}_t \times \ln(1+\mathrm{Dist}_i)\bigr) + \varepsilon_{it}
$$

```text
Y ~ Post * Dist + C(year_quarter) + C(yishuv_stat_2022)
```

`Post` is absorbed by quarter FE and $\ln(1+\mathrm{Dist})$ by SA FE. $\beta_1$
is identified from the within-SA pre/post change scaled by $\ln(1+\mathrm{Dist})$.
Positive $\beta_1$ on local-share $Y$ = stronger post-period localisation farther
from the CBD.


In [ ]:
def interaction_term(params_index, *factors):
    """Patsy name of an interaction (factor order may vary)."""
    want = set(factors)
    for name in params_index:
        if set(name.split(":")) == want:
            return name
    raise KeyError(f"interaction {want} not in {list(params_index)}")


def two_way_term(params_index, a, b):
    """Alias for interaction_term (two factors)."""
    return interaction_term(params_index, a, b)


def triple_term(params_index, a, b, c):
    """Alias for interaction_term (three factors)."""
    return interaction_term(params_index, a, b, c)


def _stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""


def _formula_lhs(formula):
    return formula.split("~", 1)[0].strip()


def _lhs_base_col(lhs):
    """Column name inside np.log1p(y) / plain y for dropna."""
    s = lhs.strip()
    if s.startswith("np.log1p(") and s.endswith(")"):
        return s[len("np.log1p("):-1].strip()
    return s


def _auto_report(params_index, fe_prefixes=("C(year_quarter)", "C(yishuv_stat_2022)")):
    """Collect Post:… interactions that are not FE dummies."""
    rows = []
    for name in params_index:
        if ":" not in name or "Post" not in name.split(":"):
            continue
        if any(name.startswith(p) for p in fe_prefixes):
            continue
        parts = tuple(name.split(":"))
        if "Post" not in parts:
            continue
        rows.append(parts)
    rows = sorted(rows, key=lambda t: (len(t), ":".join(sorted(t))))
    return rows


def fit_wls(
    panel,
    formula,
    report=None,
    *,
    transform=None,
    weights_col="n_workers_pre",
    cluster_col="yishuv_stat_2022",
    fe_cols=("year_quarter", "yishuv_stat_2022"),
):
    """Universal clustered WLS. Pass formula + which interactions to extract.

    report: list of factor tuples, e.g. [("Post", "Dist"), ("Post", "Dist", "D")].
    If None, auto-collect every Patsy name containing Post and ':'.
    transform: optional {col: callable} applied on a copy before fit
    (e.g. {"Dist": np.log1p}).
    """
    d = panel.copy()
    if transform:
        for col, fn in transform.items():
            d[col] = fn(pd.to_numeric(d[col], errors="coerce"))

    lhs = _formula_lhs(formula)
    y_col = _lhs_base_col(lhs)

    need = {y_col, "Post", weights_col, cluster_col, *fe_cols}
    if report:
        for factors in report:
            need.update(factors)
    for col in ("Dist", "log_dist"):
        if col in d.columns and (col in formula or (report and any(col in f for f in report))):
            need.add(col)

    need = [c for c in need if c in d.columns or c == y_col]
    missing = [c for c in need if c not in d.columns]
    if missing:
        raise KeyError(f"fit_wls: panel missing columns {missing}")

    # Occupation / WFH_pot: CBS missing → 0 so N matches Model 1/3 (no row drop).
    for col in ("WFH_pot", "it_pct", "finance_pct", "academic_occup_pct",
                "academic_degree_pct", "occ_it_fin_acad"):
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce").fillna(0.0)

    d = d.dropna(subset=need).copy()
    if d.empty or (d["Post"].nunique() < 2):
        return None
    post_vals = set(pd.to_numeric(d["Post"], errors="coerce").dropna().unique())
    if not post_vals.issubset({0, 1}):
        return None

    for c in fe_cols:
        if c in d.columns:
            d[c] = d[c].astype("object")
    d["Post"] = d["Post"].astype(int)
    d[y_col] = pd.to_numeric(d[y_col], errors="coerce")
    numeric_cols = set()
    if report:
        for factors in report:
            numeric_cols.update(factors)
    for col in ("Dist", "log_dist"):
        if col in d.columns and col in formula:
            numeric_cols.add(col)
    numeric_cols.discard("Post")
    for col in numeric_cols:
        if col in d.columns:
            if col in ("WFH_pot", "it_pct", "finance_pct", "academic_occup_pct",
                       "academic_degree_pct", "occ_it_fin_acad"):
                d[col] = pd.to_numeric(d[col], errors="coerce").fillna(0.0)
            else:
                d[col] = pd.to_numeric(d[col], errors="coerce")

    drop_again = [c for c in [y_col, weights_col, *numeric_cols] if c in d.columns]
    d = d.dropna(subset=drop_again)
    if d.empty or d["Post"].nunique() < 2:
        return None

    w = pd.to_numeric(d[weights_col], errors="coerce").astype(float)
    res = smf.wls(formula, data=d, weights=w).fit(
        cov_type="cluster",
        cov_kwds={"groups": d[cluster_col]},
    )

    report_tuples = list(report) if report is not None else _auto_report(res.params.index)
    interactions = {}
    for factors in report_tuples:
        key = tuple(factors)
        term = interaction_term(res.params.index, *factors)
        interactions[key] = {
            "term": term,
            "coef": float(res.params[term]),
            "se": float(res.bse[term]),
            "pval": float(res.pvalues[term]),
        }

    return {
        "formula": formula,
        "result": res,
        "report": report_tuples,
        "interactions": interactions,
        "n": int(res.nobs),
        "n_sa": int(d[cluster_col].nunique()),
        "r2": float(res.rsquared),
        "r2_adj": float(res.rsquared_adj),
        "data": d,
    }


def ix(fit, *factors):
    """Shorthand: fit['interactions'][factors]."""
    return fit["interactions"][tuple(factors)]


def print_betas(fit, beta_map, title=None):
    """Print reported terms with paper labels (β1, β2, …)."""
    if title:
        print(title)
    if fit is None:
        print("  (not estimated)")
        return
    for factors, blabel in beta_map:
        info = ix(fit, *factors)
        print(
            f"  {blabel}:  coef = {info['coef']: .6g}   "
            f"SE = {info['se']:.6g}   p = {info['pval']:.4g} {_stars(info['pval'])}"
        )
    print(f"  R² = {fit['r2']:.4f}   adj. R² = {fit['r2_adj']:.4f}")


def print_interactions(fit, title=None):
    """Pretty-print reported two-ways / triples from fit_wls."""
    if fit is None:
        print("(not estimated)")
        return
    beta_map = [(factors, " × ".join(factors)) for factors in fit["report"]]
    print_betas(fit, beta_map, title=title)


MOBILITY_OUTCOMES = ["rg_mean", "rg_poi_mean", "entropy_local_mean"]
MOBILITY_LABELS = {
    "rg_mean": "Radius of gyration, all dwells (m)",
    "rg_poi_mean": "Radius of gyration, named POIs (m)",
    "entropy_local_mean": "Local spatial entropy (nats)",
}
ALL_OUTCOMES = SHARE_OUTCOMES + MOBILITY_OUTCOMES
OUTCOME_LABELS = {**SHARE_LABELS, **MOBILITY_LABELS}
DOSE_COLS = ("IHS_Cand", "IHS_TP", "IHS_ATM", "TP_Ratio")
FE = "C(year_quarter) + C(yishuv_stat_2022)"
LOG_DIST = {"Dist": np.log1p}

missing = [c for c in ALL_OUTCOMES if c not in did.columns]
if missing:
    raise KeyError(f"Panel is missing outcome columns {missing}. Re-run Step 3.")
missing_dose = [c for c in DOSE_COLS if c not in did.columns]
if missing_dose:
    raise KeyError(f"Panel is missing dose columns {missing_dose}. Re-run Part 0 and Step 3.")
missing_dose_sa = [c for c in DOSE_COLS_SA if c not in did.columns]
if missing_dose_sa:
    raise KeyError(
        f"Panel is missing appendix dose columns {missing_dose_sa}. Re-run Part 0 and Step 3."
    )

WFH_OCC_COLS = ["it_pct"]  # continuous WFH capacity
missing_wfh = [c for c in WFH_OCC_COLS if c not in did.columns]
if missing_wfh:
    raise KeyError(f"{missing_wfh} missing on did. Re-run Step 3.")
_wfh_parts = did[WFH_OCC_COLS].apply(pd.to_numeric, errors="coerce")
n_wfh_filled = int(
    did.loc[_wfh_parts.isna().any(axis=1), "yishuv_stat_2022"].nunique()
)
# Missing CBS IT share → 0 so Models 2/4/5 keep the Model 3 sample.
did["WFH_pot"] = _wfh_parts.fillna(0.0)["it_pct"]
did["occ_it_fin_acad"] = (
    did[["it_pct", "finance_pct", "academic_occup_pct"]]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .sum(axis=1)
)

print("Step 4 grid")
print("  outcomes:", ALL_OUTCOMES)
print("  doses (WLS, log1p):", DOSE_COLS)
print("  doses appendix (SA-polygon halo):", DOSE_COLS_SA)
print("  Dist transform: ln(1+Dist) via np.log1p")
sa_wfh = did.drop_duplicates("yishuv_stat_2022")["WFH_pot"]
print("  WFH_pot = it_pct (missing → 0); continuous in Models 2–5")
print(sa_wfh.describe().round(2).rename("WFH_pot | SA"))

In [ ]:
# Model 1 — pure donut. β1 = ΔY per ln(1+Dist).
m1_rows, m1_results = [], {}
beta_m1 = [(("Post", "Dist"), "β1  Post × ln(1+Dist)")]
for y in ALL_OUTCOMES:
    formula = f"{y} ~ Post * Dist + {FE}"
    fit = fit_wls(did, formula, report=[("Post", "Dist")], transform=LOG_DIST)
    if fit is None:
        print(f"SKIP Model 1 {y}: empty after dropna")
        continue
    info = ix(fit, "Post", "Dist")
    rec = {
        "outcome": y, "label": OUTCOME_LABELS[y],
        "term": info["term"], "coef": info["coef"], "se": info["se"], "pval": info["pval"],
        "n": fit["n"], "n_sa": fit["n_sa"], "r2": fit["r2"], "r2_adj": fit["r2_adj"],
        "formula": fit["formula"], "fit": fit,
    }
    m1_results[y] = rec
    m1_rows.append({k: rec[k] for k in (
        "outcome", "label", "term", "coef", "se", "pval", "n", "n_sa", "r2", "r2_adj"
    )})

m1_table = pd.DataFrame(m1_rows)
m1_table["stars"] = m1_table["pval"].map(_stars)
print("=" * 88)
print("Model 1 — Baseline spatial shift (pure donut)")
print("Y ~ Post * Dist + C(year_quarter) + C(yishuv_stat_2022)")
print("Dist = ln(1+km). WLS weights = n_workers_pre; cluster at SA.")
print("=" * 88)
print(m1_table.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print()
for y, rec in m1_results.items():
    print_betas(rec["fit"], beta_m1, title=f"{OUTCOME_LABELS[y]}  ({y})")

In [ ]:
# Model 1 forest plot in comparable units: β × sd(ln(1+Dist)) / sd(Y).
# Raw coefficients mix duration shares (~0–1) with gyration (metres) and cannot share an axis.


def _std_beta(fit, *factors, x_col="Dist"):
    """Fully standardized interaction: SDs of Y per 1 SD of x (estimation sample)."""
    d = fit["data"]
    y_col = fit["formula"].split("~", 1)[0].strip()
    info = ix(fit, *factors)
    sx = float(pd.to_numeric(d[x_col], errors="coerce").std())
    sy = float(pd.to_numeric(d[y_col], errors="coerce").std())
    if not np.isfinite(sx) or not np.isfinite(sy) or sy == 0:
        return None
    k = sx / sy
    return {
        "std_coef": info["coef"] * k,
        "std_se": info["se"] * k,
        "sd_x": sx,
        "sd_y": sy,
    }


m1_std_rows = []
for y, rec in m1_results.items():
    std = _std_beta(rec["fit"], "Post", "Dist")
    if std is None:
        print(f"SKIP plot {y}: could not standardize")
        continue
    m1_std_rows.append({
        "outcome": y,
        "label": rec["label"],
        "family": "Duration share" if y in SHARE_OUTCOMES else "Mobility",
        **std,
        "pval": rec["pval"],
    })
m1_std = pd.DataFrame(m1_std_rows)
m1_std["outcome"] = pd.Categorical(m1_std["outcome"], categories=ALL_OUTCOMES, ordered=True)
m1_std = m1_std.sort_values("outcome").reset_index(drop=True)

PALETTE_FAM = {"Duration share": "#2b6cb0", "Mobility": "#c05621"}
fig, ax = plt.subplots(figsize=(8.4, 5.4))
ypos = np.arange(len(m1_std))
ax.axvline(0, ls="--", lw=1, color="#4a5568")
for fam, color in PALETTE_FAM.items():
    d = m1_std[m1_std["family"] == fam]
    ax.errorbar(
        d["std_coef"], d.index.to_numpy(),
        xerr=1.96 * d["std_se"],
        fmt="o", ms=6, color=color, ecolor=color,
        elinewidth=1.4, capsize=3, label=fam,
    )
ax.set_yticks(ypos)
ax.set_yticklabels(m1_std["label"])
ax.set_ylim(len(m1_std) - 0.5, -0.5)
ax.set_xlabel("Standardized β1 (SDs of Y per 1 SD of ln(1+Dist))")
ax.set_title("Model 1: post-period change by distance to the CBD")
ax.legend(frameon=False, loc="best")
fig.tight_layout()
out = _nb_dir / "3302_fig3_donut_post_dist.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out}")
print(
    "Fig. 3. Model 1 (pure donut): fully standardized WLS coefficients on "
    "Post × ln(1+Dist). Each point is β1 × sd(ln(1+Dist)) / sd(Y) on the "
    "estimation sample, so duration shares and mobility means (metres, nats) "
    "are on one axis. Horizontal bars are 95% cluster-robust intervals "
    "(cluster = statistical area). A positive value means a 1 SD increase in "
    "ln(1+Dist) is associated with a larger post-period increase in Y, in SDs "
    "of Y. Notes: Daytime weekday panel, Jan 2020 and 2023Q1; WLS weights = "
    "n_workers_pre; year-quarter and SA fixed effects. Raw (unstandardized) "
    "coefficients remain in m1_table."
)
print()
print(
    m1_std[["outcome", "family", "std_coef", "std_se", "sd_x", "sd_y", "pval"]]
    .to_string(index=False, float_format=lambda v: f"{v:.4g}")
)


### Model 2 — Occupational capacity (demographic sorting)

Does the ability to work from home drive local retention, independently of the
urban-core WFH shift ($\beta_2$) and suburban remote-work flight ($\beta_3$)?

$$
\begin{aligned}
Y_{it} = \alpha_i + \gamma_{q(t)}
&+ \beta_1\bigl(\mathrm{Post}_t \times \ln(1+\mathrm{Dist}_i)\bigr) \\
&+ \beta_2\bigl(\mathrm{Post}_t \times \mathrm{WFH}_i\bigr)
+ \beta_3\bigl(\mathrm{Post}_t \times \ln(1+\mathrm{Dist}_i) \times \mathrm{WFH}_i\bigr)
+ \varepsilon_{it}
\end{aligned}
$$

```text
Y ~ Post * Dist * WFH_pot + C(year_quarter) + C(yishuv_stat_2022)
```

`WFH_pot` is the CBS 2022 IT employment share (`it_pct`; missing coded 0),
entered continuously. SA FE absorb $\ln(1+\mathrm{Dist})$, `WFH_pot`, and their product.

- $\beta_1$: donut at `WFH_pot` = 0
- $\beta_2$: extra post-period shift per pp of IT share at Dist = 0 km
- $\beta_3$: extra donut steepness per pp of IT share


In [ ]:
# Model 2 — Post × Dist × WFH_pot.
beta_m2 = [
    (("Post", "Dist"), "β1  Post × ln(1+Dist)"),
    (("Post", "WFH_pot"), "β2  Post × WFH_pot"),
    (("Post", "Dist", "WFH_pot"), "β3  Post × ln(1+Dist) × WFH_pot"),
]
m2_rows, m2_results = [], {}
for y in ALL_OUTCOMES:
    formula = f"{y} ~ Post * Dist * WFH_pot + {FE}"
    fit = fit_wls(
        did, formula,
        report=[("Post", "Dist"), ("Post", "WFH_pot"), ("Post", "Dist", "WFH_pot")],
        transform=LOG_DIST,
    )
    if fit is None:
        print(f"SKIP Model 2 {y}: empty after dropna")
        continue
    b1, b2, b3 = ix(fit, "Post", "Dist"), ix(fit, "Post", "WFH_pot"), ix(fit, "Post", "Dist", "WFH_pot")
    rec = {
        "outcome": y, "label": OUTCOME_LABELS[y],
        "n": fit["n"], "n_sa": fit["n_sa"], "r2": fit["r2"], "r2_adj": fit["r2_adj"],
        "fit": fit,
        "b1": b1["coef"], "b1_se": b1["se"], "b1_p": b1["pval"],
        "b2": b2["coef"], "b2_se": b2["se"], "b2_p": b2["pval"],
        "b3": b3["coef"], "b3_se": b3["se"], "b3_p": b3["pval"],
    }
    m2_results[y] = rec
    m2_rows.append(rec)

m2_table = pd.DataFrame(m2_rows)
print("=" * 88)
print("Model 2 — Occupational capacity (demographic sorting)")
print("Y ~ Post * Dist * WFH_pot + C(year_quarter) + C(yishuv_stat_2022)")
print("  β1  Post × ln(1+Dist)                 (donut at WFH_pot = 0)")
print("  β2  Post × WFH_pot                    (core capacity, Dist = 0 km)")
print("  β3  Post × ln(1+Dist) × WFH_pot       (suburban remote-work flight)")
print("=" * 88)
show = m2_table[["outcome", "label", "b1", "b1_se", "b1_p", "b2", "b2_se", "b2_p", "b3", "b3_se", "b3_p", "n", "n_sa", "r2", "r2_adj"]]
print(show.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print()
for rec in m2_rows:
    print_betas(rec["fit"], beta_m2, title=f"\n{rec['label']}  ({rec['outcome']})")


### Model 3 — 15-minute city test (amenity pull)

Run sequentially for each amenity dose (`IHS_Cand`, `IHS_TP`, `IHS_ATM`,
`TP_Ratio`). `IHS_*` = `log(1 +` home-disc mean`)`.
`TP_Ratio = TP_n / (Cand_n + 1)` is the detected share of nearby candidates.
Raw counts stay on `df_sa`. Isolates core pull ($\beta_4$) and suburban
amenity failure / amplification ($\beta_5$).
($\beta_4$) and suburban amenity failure / amplification ($\beta_5$).

$$
\begin{aligned}
Y_{it} = \alpha_i + \gamma_{q(t)}
&+ \beta_1\bigl(\mathrm{Post}_t \times \ln(1+\mathrm{Dist}_i)\bigr) \\
&+ \beta_4\bigl(\mathrm{Post}_t \times \mathrm{Dose}_i\bigr)
+ \beta_5\bigl(\mathrm{Post}_t \times \ln(1+\mathrm{Dist}_i) \times \mathrm{Dose}_i\bigr)
+ \varepsilon_{it}
\end{aligned}
$$

```text
Y ~ Post * Dist * Dose + C(year_quarter) + C(yishuv_stat_2022)
```

- $\beta_1$: donut at Dose = 0
- $\beta_4$: amenity slope at Dist = 0 km (core pull)
- $\beta_5$: how the donut steepens with walkable stock (suburban amenity test)


In [ ]:
# Model 3 — Post × Dist × Dose, one amenity type at a time.
m3_rows, m3_results = [], {}
print("=" * 88)
print("Model 3 — 15-minute city (amenity pull; home-1 km mean)")
print("Y ~ Post * Dist * Dose + C(year_quarter) + C(yishuv_stat_2022)")
print("  β1  Post × ln(1+Dist)                 (donut at Dose = 0)")
print("  β4  Post × Dose                       (core amenity pull, Dist = 0 km)")
print("  β5  Post × ln(1+Dist) × Dose          (suburban amenity test)")
for i, dose in enumerate(DOSE_COLS, 1):
    print(f"Dose {i}: {dose}")
print("=" * 88)

for y in ALL_OUTCOMES:
    for dose in DOSE_COLS:
        formula = f"{y} ~ Post * Dist * {dose} + {FE}"
        report = [("Post", "Dist"), ("Post", dose), ("Post", "Dist", dose)]
        fit = fit_wls(did, formula, report=report, transform=LOG_DIST)
        if fit is None:
            print(f"SKIP Model 3 {y} × {dose}: empty after dropna")
            continue
        b1 = ix(fit, "Post", "Dist")
        b4 = ix(fit, "Post", dose)
        b5 = ix(fit, "Post", "Dist", dose)
        rec = {
            "outcome": y, "label": OUTCOME_LABELS[y], "dose": dose,
            "n": fit["n"], "n_sa": fit["n_sa"], "r2": fit["r2"], "r2_adj": fit["r2_adj"],
            "fit": fit,
            "b1": b1["coef"], "b1_se": b1["se"], "b1_p": b1["pval"],
            "b4": b4["coef"], "b4_se": b4["se"], "b4_p": b4["pval"],
            "b5": b5["coef"], "b5_se": b5["se"], "b5_p": b5["pval"],
        }
        m3_results[(y, dose)] = rec
        m3_rows.append(rec)

m3_table = pd.DataFrame(m3_rows)
print(m3_table[[
    "outcome", "label", "dose",
    "b1", "b1_se", "b1_p", "b4", "b4_se", "b4_p", "b5", "b5_se", "b5_p", "n", "n_sa", "r2", "r2_adj",
]].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print()
for y in ALL_OUTCOMES:
    print(f"\n{OUTCOME_LABELS[y]}  ({y})")
    for dose in DOSE_COLS:
        rec = m3_results.get((y, dose))
        if rec is None:
            print(f"  {dose}: (not estimated)")
            continue
        beta_m3 = [
            (("Post", "Dist"), "β1  Post × ln(1+Dist)"),
            (("Post", dose), f"β4  Post × {dose}"),
            (("Post", "Dist", dose), f"β5  Post × ln(1+Dist) × {dose}"),
        ]
        print_betas(rec["fit"], beta_m3, title=f"  Dose = {dose}")


### Model 4 — Additive horse race (capacity vs attraction)

Combines Models 2 and 3 so occupational WFH capacity and amenities compete.
Asks whether third-place effects survive after holding continuous IT share fixed.

$$
\begin{aligned}
Y_{it} = \alpha_i + \gamma_{q(t)}
&+ \beta_1\bigl(\mathrm{Post}_t \times \ln(1+\mathrm{Dist}_i)\bigr) \\
&+ \beta_2\bigl(\mathrm{Post}_t \times \mathrm{WFH}_i\bigr)
+ \beta_3\bigl(\mathrm{Post}_t \times \ln(1+\mathrm{Dist}_i) \times \mathrm{WFH}_i\bigr) \\
&+ \beta_4\bigl(\mathrm{Post}_t \times \mathrm{Dose}_i\bigr)
+ \beta_5\bigl(\mathrm{Post}_t \times \ln(1+\mathrm{Dist}_i) \times \mathrm{Dose}_i\bigr)
+ \varepsilon_{it}
\end{aligned}
$$

```text
Y ~ Post * Dist * WFH_pot + Post * Dist * Dose + C(year_quarter) + C(yishuv_stat_2022)
```

Same $\beta$ numbering as Models 2–3. $\beta_4$ and $\beta_5$ are the amenity
terms *conditional on* WFH_pot × Dist; $\beta_2$ and $\beta_3$ are WFH_pot terms
*conditional on* Dose × Dist.


In [ ]:
# Model 4 — additive horse race: WFH and Dose both interact with Post × Dist.
m4_rows, m4_results = [], {}
print("=" * 88)
print("Model 4 — Additive horse race (WFH capacity vs home-disc amenity pull)")
print("Y ~ Post * Dist * WFH_pot + Post * Dist * Dose + C(year_quarter) + C(yishuv_stat_2022)")
print("  β1  Post × ln(1+Dist)")
print("  β2  Post × WFH_pot                    β3  Post × ln(1+Dist) × WFH_pot")
print("  β4  Post × Dose                       β5  Post × ln(1+Dist) × Dose")
print("=" * 88)

for y in ALL_OUTCOMES:
    for dose in DOSE_COLS:
        formula = f"{y} ~ Post * Dist * WFH_pot + Post * Dist * {dose} + {FE}"
        report = [
            ("Post", "Dist"),
            ("Post", "WFH_pot"),
            ("Post", "Dist", "WFH_pot"),
            ("Post", dose),
            ("Post", "Dist", dose),
        ]
        fit = fit_wls(did, formula, report=report, transform=LOG_DIST)
        if fit is None:
            print(f"SKIP Model 4 {y} × {dose}: empty after dropna")
            continue
        b1 = ix(fit, "Post", "Dist")
        b2 = ix(fit, "Post", "WFH_pot")
        b3 = ix(fit, "Post", "Dist", "WFH_pot")
        b4 = ix(fit, "Post", dose)
        b5 = ix(fit, "Post", "Dist", dose)
        rec = {
            "outcome": y, "label": OUTCOME_LABELS[y], "dose": dose,
            "n": fit["n"], "n_sa": fit["n_sa"], "r2": fit["r2"], "r2_adj": fit["r2_adj"],
            "fit": fit,
            "b1": b1["coef"], "b1_se": b1["se"], "b1_p": b1["pval"],
            "b2": b2["coef"], "b2_se": b2["se"], "b2_p": b2["pval"],
            "b3": b3["coef"], "b3_se": b3["se"], "b3_p": b3["pval"],
            "b4": b4["coef"], "b4_se": b4["se"], "b4_p": b4["pval"],
            "b5": b5["coef"], "b5_se": b5["se"], "b5_p": b5["pval"],
        }
        m4_results[(y, dose)] = rec
        m4_rows.append(rec)

m4_table = pd.DataFrame(m4_rows)
print(m4_table[[
    "outcome", "label", "dose",
    "b1", "b1_p", "b2", "b2_p", "b3", "b3_p", "b4", "b4_p", "b5", "b5_p", "n", "n_sa", "r2", "r2_adj",
]].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print()
for y in ALL_OUTCOMES:
    print(f"\n{OUTCOME_LABELS[y]}  ({y})")
    for dose in DOSE_COLS:
        rec = m4_results.get((y, dose))
        if rec is None:
            print(f"  {dose}: (not estimated)")
            continue
        beta_m4 = [
            (("Post", "Dist"), "β1  Post × ln(1+Dist)"),
            (("Post", "WFH_pot"), "β2  Post × WFH_pot"),
            (("Post", "Dist", "WFH_pot"), "β3  Post × ln(1+Dist) × WFH_pot"),
            (("Post", dose), f"β4  Post × {dose}"),
            (("Post", "Dist", dose), f"β5  Post × ln(1+Dist) × {dose}"),
        ]
        print_betas(rec["fit"], beta_m4, title=f"  Dose = {dose}")


### Model 5 — Synergy test

Do physical amenities and remote-work capacity multiply each other? Dist is
*not* in this equation — the donut is already in Models 1–4. Following
marginality, `Post * WFH_pot * Dose` includes every lower-order term, so
$\beta_6$ is the extra post-period shift when higher IT share and high amenity
stock coincide.

$$
\begin{aligned}
Y_{it} = \alpha_i + \gamma_{q(t)}
&+ \beta_2\bigl(\mathrm{Post}_t \times \mathrm{WFH}_i\bigr)
+ \beta_4\bigl(\mathrm{Post}_t \times \mathrm{Dose}_i\bigr) \\
&+ \beta_6\bigl(\mathrm{Post}_t \times \mathrm{WFH}_i \times \mathrm{Dose}_i\bigr)
+ \varepsilon_{it}
\end{aligned}
$$

```text
Y ~ Post * WFH_pot * Dose + C(year_quarter) + C(yishuv_stat_2022)
```

SA FE absorb all time-invariant products (WFH_pot, Dose, and their cross).
$\beta_2$ and $\beta_4$ are the same objects as in Models 2–4 but without Dist
interactions. $\beta_6 > 0$ on local-share $Y$ means amenities raise post-period
local time *more* where IT share is higher.


In [ ]:
# Model 5 — synergy: Post × WFH_pot × Dose (no Dist).
m5_rows, m5_results = [], {}
print("=" * 88)
print("Model 5 — Synergy test (Post × WFH_pot × Dose)")
print("Y ~ Post * WFH_pot * Dose + C(year_quarter) + C(yishuv_stat_2022)")
print("  β2  Post × WFH_pot")
print("  β4  Post × Dose")
print("  β6  Post × WFH_pot × Dose             (synergy)")
print("=" * 88)

for y in ALL_OUTCOMES:
    for dose in DOSE_COLS:
        formula = f"{y} ~ Post * WFH_pot * {dose} + {FE}"
        report = [
            ("Post", "WFH_pot"),
            ("Post", dose),
            ("Post", "WFH_pot", dose),
        ]
        fit = fit_wls(did, formula, report=report, transform=LOG_DIST)
        if fit is None:
            print(f"SKIP Model 5 {y} × {dose}: empty after dropna")
            continue
        b2 = ix(fit, "Post", "WFH_pot")
        b4 = ix(fit, "Post", dose)
        b6 = ix(fit, "Post", "WFH_pot", dose)
        rec = {
            "outcome": y, "label": OUTCOME_LABELS[y], "dose": dose,
            "n": fit["n"], "n_sa": fit["n_sa"], "r2": fit["r2"], "r2_adj": fit["r2_adj"],
            "fit": fit,
            "b2": b2["coef"], "b2_se": b2["se"], "b2_p": b2["pval"],
            "b4": b4["coef"], "b4_se": b4["se"], "b4_p": b4["pval"],
            "b6": b6["coef"], "b6_se": b6["se"], "b6_p": b6["pval"],
        }
        m5_results[(y, dose)] = rec
        m5_rows.append(rec)

m5_table = pd.DataFrame(m5_rows)
print(m5_table[[
    "outcome", "label", "dose",
    "b2", "b2_p", "b4", "b4_p", "b6", "b6_p",
    "n", "n_sa", "r2", "r2_adj",
]].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print()
for y in ALL_OUTCOMES:
    print(f"\n{OUTCOME_LABELS[y]}  ({y})")
    for dose in DOSE_COLS:
        rec = m5_results.get((y, dose))
        if rec is None:
            print(f"  {dose}: (not estimated)")
            continue
        beta_m5 = [
            (("Post", "WFH_pot"), "β2  Post × WFH_pot"),
            (("Post", dose), f"β4  Post × {dose}"),
            (("Post", "WFH_pot", dose), f"β6  Post × WFH_pot × {dose}"),
        ]
        print_betas(rec["fit"], beta_m5, title=f"  Dose = {dose}")


### Appendix — SA-polygon neighbourhood supply

Robustness only. Same Models 3–4, replacing the home-disc doses with PU counts
inside `ST_DWITHIN(sa_geom, 500 m)` (`IHS_Cand_sa`, `IHS_TP_sa`, `IHS_ATM_sa`).
That is neighbourhood *supply around the SA polygon*, not the 1 km disc around
each worker home. Do not put home-disc and polygon-halo doses in the same
equation. Local is still `dist_to_home < 1 km`.

In [ ]:
# Appendix: Models 3–4 with SA-polygon + 500 m halo doses (not the main spec).
m3_sa_rows, m3_sa_results = [], {}
print("=" * 88)
print("Appendix Model 3 — SA-polygon neighbourhood supply")
print("Y ~ Post * Dist * Dose_sa + C(year_quarter) + C(yishuv_stat_2022)")
print("  Dose_sa ∈", DOSE_COLS_SA)
print("=" * 88)

for y in ALL_OUTCOMES:
    for dose in DOSE_COLS_SA:
        formula = f"{y} ~ Post * Dist * {dose} + {FE}"
        report = [("Post", "Dist"), ("Post", dose), ("Post", "Dist", dose)]
        fit = fit_wls(did, formula, report=report, transform=LOG_DIST)
        if fit is None:
            print(f"SKIP Appendix M3 {y} × {dose}: empty after dropna")
            continue
        b1 = ix(fit, "Post", "Dist")
        b4 = ix(fit, "Post", dose)
        b5 = ix(fit, "Post", "Dist", dose)
        rec = {
            "outcome": y, "label": OUTCOME_LABELS[y], "dose": dose,
            "n": fit["n"], "n_sa": fit["n_sa"], "r2": fit["r2"], "r2_adj": fit["r2_adj"],
            "fit": fit,
            "b1": b1["coef"], "b1_se": b1["se"], "b1_p": b1["pval"],
            "b4": b4["coef"], "b4_se": b4["se"], "b4_p": b4["pval"],
            "b5": b5["coef"], "b5_se": b5["se"], "b5_p": b5["pval"],
        }
        m3_sa_results[(y, dose)] = rec
        m3_sa_rows.append(rec)

m3_sa_table = pd.DataFrame(m3_sa_rows)
print(m3_sa_table[[
    "outcome", "label", "dose",
    "b1", "b1_se", "b1_p", "b4", "b4_se", "b4_p", "b5", "b5_se", "b5_p", "n", "n_sa", "r2", "r2_adj",
]].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

m4_sa_rows, m4_sa_results = [], {}
print("=" * 88)
print("Appendix Model 4 — horse race with SA-polygon doses")
print("Y ~ Post * Dist * WFH_pot + Post * Dist * Dose_sa + FE")
print("=" * 88)

for y in ALL_OUTCOMES:
    for dose in DOSE_COLS_SA:
        formula = f"{y} ~ Post * Dist * WFH_pot + Post * Dist * {dose} + {FE}"
        report = [
            ("Post", "Dist"),
            ("Post", "WFH_pot"),
            ("Post", "Dist", "WFH_pot"),
            ("Post", dose),
            ("Post", "Dist", dose),
        ]
        fit = fit_wls(did, formula, report=report, transform=LOG_DIST)
        if fit is None:
            print(f"SKIP Appendix M4 {y} × {dose}: empty after dropna")
            continue
        b1 = ix(fit, "Post", "Dist")
        b2 = ix(fit, "Post", "WFH_pot")
        b3 = ix(fit, "Post", "Dist", "WFH_pot")
        b4 = ix(fit, "Post", dose)
        b5 = ix(fit, "Post", "Dist", dose)
        rec = {
            "outcome": y, "label": OUTCOME_LABELS[y], "dose": dose,
            "n": fit["n"], "n_sa": fit["n_sa"], "r2": fit["r2"], "r2_adj": fit["r2_adj"],
            "fit": fit,
            "b1": b1["coef"], "b1_se": b1["se"], "b1_p": b1["pval"],
            "b2": b2["coef"], "b2_se": b2["se"], "b2_p": b2["pval"],
            "b3": b3["coef"], "b3_se": b3["se"], "b3_p": b3["pval"],
            "b4": b4["coef"], "b4_se": b4["se"], "b4_p": b4["pval"],
            "b5": b5["coef"], "b5_se": b5["se"], "b5_p": b5["pval"],
        }
        m4_sa_results[(y, dose)] = rec
        m4_sa_rows.append(rec)

m4_sa_table = pd.DataFrame(m4_sa_rows)
print(m4_sa_table[[
    "outcome", "label", "dose",
    "b4", "b4_se", "b4_p", "b5", "b5_se", "b5_p", "n", "n_sa", "r2", "r2_adj",
]].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

## Reading the estimates

**Model 1 (pure donut).** $\beta_1$ (`Post × ln(1+Dist)`) is suburbanisation of
daytime activity, ignoring local amenities and occupations. Positive on local
shares = more localisation farther from the CBD after 2020.

**Model 2 (occupational capacity).** $\beta_2$ is the extra post-period shift per
pp of IT share (`WFH_pot` = `it_pct`) at Dist = 0 km. $\beta_3$ is how that donut
steepens with IT share (suburban remote-work flight). $\beta_1$ is the donut
at `WFH_pot` = 0.

**Model 3 (amenity pull).** Same geometry with Dose in place of WFH. $\beta_4$
is core amenity retention; $\beta_5$ is whether walkable stock steepens (or
flattens) the suburban donut. Run separately for `IHS_Cand`, `IHS_TP`,
`IHS_ATM`, and `TP_Ratio` (`TP_n / (Cand_n + 1)`).

**Model 3.1 (joint Cand vs TP).** `IHS_Cand` and `IHS_TP` in one equation so
detected intensity is net of candidate volume.
(home-1 km means).

**Model 4 (additive horse race).** Models 2 and 3 in one equation. Read $\beta_4$
and $\beta_5$ as amenity effects *net of* WFH_pot × Dist; read $\beta_2$ and
$\beta_3$ as WFH_pot effects *net of* Dose × Dist. If $\beta_5$ stays signed and
precise, third-place intensity is not just a proxy for high-tech workers.

**Model 5 (synergy).** `Post * WFH_pot * Dose` — Dist is omitted. $\beta_6$
(`Post × WFH_pot × Dose`) is synergy: the extra post-period shift when higher
IT share and high amenity stock coincide. $\beta_2$ and $\beta_4$ are WFH and
dose main post-shifts without Dist interactions. All time-invariant crosses
are absorbed by SA FE.

- **Local shares** (`share_local_*`): positive $\beta_6$ means an extra unit of
  dose × WFH_pot raises local time more after remote work.
- **`share_home_nonwork_h`**: home hours among non-work hours — a WFH
  diagnostic, not neighbourhood retention.
- **`share_local_nothome_nonwork_h`**: local hours among time that is neither
  home nor work.
- **`share_poi_of_local_h`**: named local POI hours among local weekday-daytime
  hours. Dual of `share_local_poi_h`.
- **`rg_mean` / `rg_poi_mean`**: negative coefficients = the activity cloud
  shrinks.
- **Walkable stock.** `IHS_Cand`, `IHS_TP`, `IHS_ATM` are `log(1 +` SA-mean
  venue counts within 1 km of January 2020 home geohashes`)`. `TP_Ratio` =
  `TP_n / (Cand_n + 1)`. ATM is atmosphere-suitable **or** library.
  Polygon-halo (`IHS_*_sa`) is appendix only.

`Post` is absorbed by `C(year_quarter)`. SA FE absorb time-invariant SA levels
(`Dist`, WFH, dose, and locality). Identifying variation is the within-SA
change between Jan 2020 and 2023Q1, scaled by the interactions above.
